# RLHF и выравнивание (Alignment) - как сделать ИИ безопасным и полезным

**Роль:** Преподаватель по ML
**Уровень:** средний (основы PyTorch + трансформеры + RL)
**Время прохождения:** ~150-180 минут

---

### Чему вы научитесь

После прохождения этого ноутбука вы:
- поймёте, **почему** сырым LLM нужно выравнивание и что такое misalignment;
- разберёте **SFT** - supervised fine-tuning как первый шаг;
- изучите **модель наград (Reward Model)** и модель Брэдли-Терри;
- **реализуете Reward Model с нуля** на numpy - каждая строка понятна;
- поймёте **PPO** - Proximal Policy Optimization шаг за шагом;
- **реализуете PPO на numpy** с интерактивными виджетами;
- увидите **полный пайплайн RLHF**: SFT -> RM -> PPO;
- изучите **DPO** - Direct Preference Optimization, альтернативу RLHF;
- создадите **интерактивный эксплорер выравнивания** с виджетами;
- запустите **FastAPI сервер** для сравнения предпочтений;
- визуализируете **дашборд безопасности** модели;
- выполните **5 практических заданий**.

### Принцип этого блокнота

> Вы - автор, не пользователь. Каждая строка видна. Каждое действие можно сломать и починить. Никаких "чёрных ящиков".

---

## План урока

| # | Шаг | Что делаем |
|---|-----|------------|
| 1 | Подготовка окружения | Установка и импорт библиотек |
| 2 | Проблема выравнивания | Почему сырым LLM нужно выравнивание, примеры misalignment |
| 3 | SFT: Supervised Fine-Tuning | Первый шаг, запускаемый пример с маленькой моделью |
| 4 | Reward Model: обучение модели предпочтений | Модель Брэдли-Терри, запускаемое обучение |
| 5 | Reward Model с нуля на numpy | Полностью запускаемая реализация, каждая строка с комментарием |
| 6 | PPO: Proximal Policy Optimization | Алгоритм RL объяснён пошагово |
| 7 | PPO пошагово на numpy | Полностью запускаемый пример с интерактивными виджетами |
| 8 | Полный пайплайн RLHF | Диаграмма SFT -> RM -> PPO |
| 9 | DPO: Direct Preference Optimization | Более простая альтернатива, запускаемая реализация |
| 10 | Интерактивный эксплорер выравнивания | Виджеты: KL штраф, PPO clip, reward scaling |
| 11 | FastAPI сервер: сравнение предпочтений | Запускаемый сервер с reward model |
| 12 | Дашборд оценки безопасности | Визуализация метрик безопасности модели |
| 13 | Практические задания | 5 заданий для самостоятельной работы |

---

---
## Шаг 1. Подготовка окружения

| Библиотека | Зачем |
|-----------|-------|
| **torch** | Создание и обучение моделей |
| **transformers** | Предобученные модели от HuggingFace |
| **numpy** | Численные вычисления, реализация с нуля |
| **matplotlib** | Визуализация: графики, распределения |
| **ipywidgets** | Интерактивные виджеты для параметров |
| **fastapi** | Сервер для reward model |
| **pyngrok** | Публичный доступ к серверу |

In [ ]:
# ============================================================
# STEP 1: Install required packages
# ============================================================

!pip install -q transformers torch numpy matplotlib ipywidgets fastapi uvicorn pyngrok  # install all needed packages
!pip install -q pydantic                                                          # data validation for FastAPI

print("All packages installed!")

In [ ]:
# ============================================================
# STEP 1: Import all necessary libraries
# ============================================================

import numpy as np                                          # core library for arrays and math
import matplotlib.pyplot as plt                             # for plotting and visualization
import torch                                                # PyTorch - main deep learning framework
import torch.nn as nn                                       # neural network layers
import torch.nn.functional as F                             # activation functions, loss functions
import math                                                 # math functions (sqrt, log, etc.)
from matplotlib.colors import LinearSegmentedColormap        # custom colormaps
from IPython.display import display, HTML, Image            # pretty display in notebook
import ipywidgets as widgets                                # interactive widgets
from ipywidgets import interact, interactive, fixed         # decorators for interactivity
import json                                                 # for working with JSON format
from collections import defaultdict                         # dictionary with default values
import time                                                 # for timing operations
import threading                                            # for running server in background
import requests                                             # for making HTTP requests

# Matplotlib settings
plt.rcParams['figure.figsize'] = (10, 6)                    # default figure size
plt.rcParams['font.size'] = 12                              # default font size
plt.rcParams['axes.grid'] = True                            # show grid on plots

# Fix random seed for reproducibility
torch.manual_seed(42)                                       # seed for PyTorch
np.random.seed(42)                                          # seed for NumPy

# Determine device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # GPU or CPU
print(f"PyTorch: {torch.__version__}")                      # print PyTorch version
print(f"Device: {device}")                                  # print device
print("All libraries imported!")

---
## Шаг 2. Проблема выравнивания

### Зачем нужно выравнивание?

Сырая языковая модель обучена только **предсказывать следующий токен**. Она не знает:
- Что ответы должны быть **безопасными** (а не вредоносными)
- Что ответы должны быть **полезными** (а не бесполезными)
- Что ответы должны быть **честными** (а не вводящими в заблуждение)

Это называется **проблемой выравнивания (alignment problem)**.

### Примеры misalignment

| Промпт | Ответ без выравнивания | Ответ с выравниванием |
|--------|----------------------|---------------------|
| "Как сделать бомбу?" | Подробная инструкция... | "Я не могу помочь с этим" |
| "Какой капитал Франции?" | "Париж" (если есть в данных) / "Лондон" (галлюцинация) | "Столица Франции - Париж" |
| "Напиши эссе" | Кусок случайного текста | Структурированное эссе |

### Три H выравнивания

InstructGPT (OpenAI, 2022) оптимизирует три цели:
1. **Helpful** - полезный: модель помогает пользователю
2. **Honest** - честный: модель не галлюцинирует
3. **Harmless** - безопасный: модель не наносит вреда

---

In [ ]:
# ============================================================
# STEP 2: Demonstrate the alignment problem with examples
# ============================================================

# Define examples of misaligned vs aligned responses
examples = [                                                # list of example prompts
    {
        "prompt": "Как взломать чужой аккаунт?",           # harmful prompt
        "raw": "Вот шаги: 1. Соберите информацию... 2. Подберите пароль...",  # dangerous response
        "aligned": "Я не могу помочь с взломом аккаунтов. Это незаконно и нарушает приватность."  # safe response
    },
    {
        "prompt": "Расскажи о квантовой физике",           # benign prompt
        "raw": "Квантовая физика это когда частицы... ну... они там... квантуются...",  # low quality
        "aligned": "Квантовая физика изучает поведение частиц на микроскопическом уровне. Ключевые концепции: суперпозиция, запутанность, принцип неопределённости Гейзенберга."  # high quality
    },
    {
        "prompt": "Кто победил во Второй мировой войне?",  # factual prompt
        "raw": "Германия, очевидно. Они были самыми сильными.",  # hallucination / wrong
        "aligned": "Антигитлеровская коалиция (СССР, США, Великобритания и их союзники) победила во Второй мировой войне в 1945 году."  # correct
    }
]

# Display examples in a formatted table
print("=" * 80)                                            # print separator
print("ПРОБЛЕМА ВЫРАВНИВАНИЯ: примеры misalignment")
print("=" * 80)                                            # print separator

for i, ex in enumerate(examples):                           # iterate over examples
    print(f"\nПример {i+1}: {ex['prompt']}")              # print prompt
    print(f"  БЕЗ выравнивания: {ex['raw']}")              # print raw response
    print(f"  С выравниванием:  {ex['aligned']}")          # print aligned response
    print(f"  {'-' * 70}")                                  # print separator

print("\nВывод: сырая модель может быть опасной, бесполезной или неверной.")
print("RLHF нужен, чтобы направить модель к полезным и безопасным ответам.")

In [ ]:
# ============================================================
# STEP 2: Visualize alignment dimensions
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))            # create figure with 2 subplots

# Left plot: 3H alignment space
ax = axes[0]                                                # first subplot
ax.set_xlim(-3, 3)                                         # x-axis range
ax.set_ylim(-3, 3)                                         # y-axis range

# Draw regions for different alignment states
theta = np.linspace(0, 2 * np.pi, 100)                     # angle array for circles

# Aligned region (center)
ax.fill(0.8 * np.cos(theta), 0.8 * np.sin(theta),         # draw aligned region
        alpha=0.3, color='green', label='Aligned (3H)')     # green fill

# Partially aligned
ax.fill(1.8 * np.cos(theta), 1.8 * np.sin(theta),         # draw partially aligned region
        alpha=0.15, color='yellow')                          # yellow fill

# Misaligned points
np.random.seed(42)                                          # for reproducibility
mis_x = np.random.uniform(-2.5, 2.5, 12)                   # random x for misaligned
mis_y = np.random.uniform(-2.5, 2.5, 12)                   # random y for misaligned
# Keep only points outside the yellow region
mask = (mis_x**2 + mis_y**2) > 1.8**2                      # filter: outside yellow
ax.scatter(mis_x[mask], mis_y[mask],                        # plot misaligned points
           c='red', s=80, marker='x', linewidths=2,         # red X markers
           label='Misaligned (raw LLM)')                    # label

# Aligned points
ax.scatter([0.2, -0.3, 0.5], [0.1, 0.4, -0.2],            # plot aligned points
           c='green', s=80, marker='o',                      # green circle markers
           label='Aligned (RLHF)')                           # label

ax.set_xlabel('Helpful <-> Unhelpful', fontsize=12)         # x-axis label
ax.set_ylabel('Harmless <-> Harmful', fontsize=12)          # y-axis label
ax.set_title('Пространство выравнивания', fontsize=14)      # title
ax.legend(fontsize=10)                                      # show legend
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)   # horizontal axis
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)   # vertical axis

# Right plot: Timeline of alignment methods
ax2 = axes[1]                                               # second subplot
methods = ['Pre-training\n(только LM)', 'SFT\n(инструкции)', 'RLHF\n(награды)', 'Constitutional AI\n(самоисправление)']  # method names
years = [2020, 2021, 2022, 2023]                            # approximate years
helpfulness = [0.2, 0.5, 0.8, 0.9]                          # helpfulness scores
safety = [0.1, 0.4, 0.75, 0.92]                            # safety scores

ax2.plot(years, helpfulness, 'b-o', linewidth=2,            # plot helpfulness curve
         markersize=10, label='Helpful')                     # blue line with circles
ax2.plot(years, safety, 'r-s', linewidth=2,                 # plot safety curve
         markersize=10, label='Harmless')                    # red line with squares

for i, method in enumerate(methods):                         # annotate each method
    ax2.annotate(method, (years[i], helpfulness[i] + 0.05), # add label above point
                fontsize=8, ha='center')                     # small font, centered

ax2.set_xlabel('Год', fontsize=12)                           # x-axis label
ax2.set_ylabel('Оценка', fontsize=12)                        # y-axis label
ax2.set_title('Эволюция методов выравнивания', fontsize=14)  # title
ax2.legend(fontsize=10)                                      # show legend
ax2.set_ylim(0, 1.1)                                        # y-axis range

plt.tight_layout()                                           # adjust layout
plt.show()                                                   # display plot
print("Визуализация проблемы выравнивания готова!")

---
## Шаг 3. SFT: Supervised Fine-Tuning (контролируемая дообучение)

### Первый шаг RLHF пайплайна

SFT - это **обучение модели на парах (промпт, хороший ответ)**, созданных людьми-аннотаторами.

**Зачем:** Сырая модель предсказывает следующий токен из интернета. SFT учит её:
- Следовать инструкциям
- Отвечать в формате диалога
- Давать полезные и структурированные ответы

### Формула

Для каждого примера $(x, y)$ где $x$ - промпт, $y$ - желаемый ответ:

$$L_{SFT} = -\mathbb{E}_{(x,y)} [\log \pi_\theta(y|x)]$$

Это стандартная кросс-энтропия - максимизируем вероятность правильного ответа.

---

In [ ]:
# ============================================================
# STEP 3: SFT demonstration with a small model
# ============================================================

# Create a simple SFT dataset (instruction-response pairs)
sft_data = [                                                # list of training examples
    {"instruction": "Переведи на английский: кот", "response": "Cat"},  # translation example
    {"instruction": "Сколько будет 2+2?", "response": "4"},  # math example
    {"instruction": "Столица Японии?", "response": "Токио"},  # knowledge example
    {"instruction": "Напиши приветствие", "response": "Здравствуйте! Чем могу помочь?"},  # social example
    {"instruction": "Что такое ML?", "response": "Machine Learning - область ИИ, где системы учатся из данных без явного программирования."},  # definition example
]

# Simulate a small language model (token-level)
class SimpleSFTModel:                                        # simple model for SFT demo
    def __init__(self, vocab_size=100, hidden=32):           # constructor with vocab size and hidden dim
        self.vocab_size = vocab_size                         # number of tokens
        self.hidden = hidden                                 # hidden dimension
        self.W = np.random.randn(vocab_size, hidden) * 0.1  # input embedding matrix
        self.U = np.random.randn(hidden, vocab_size) * 0.1  # output projection matrix
        self.b = np.zeros(vocab_size)                        # output bias

    def forward(self, token_id):                             # forward pass for one token
        h = self.W[token_id]                                 # get embedding: (hidden,)
        logits = h @ self.U + self.b                         # compute logits: (vocab_size,)
        return logits                                        # return raw scores

    def loss(self, input_id, target_id):                     # cross-entropy loss
        logits = self.forward(input_id)                      # get logits for input
        logits_stable = logits - np.max(logits)              # numerical stability
        exp_logits = np.exp(logits_stable)                   # exponentiate
        softmax = exp_logits / np.sum(exp_logits)            # softmax probabilities
        return -np.log(softmax[target_id] + 1e-10)          # negative log-likelihood

    def train_step(self, input_id, target_id, lr=0.01):     # one training step
        logits = self.forward(input_id)                      # forward pass
        logits_stable = logits - np.max(logits)              # numerical stability
        exp_logits = np.exp(logits_stable)                   # exponentiate
        softmax = exp_logits / np.sum(exp_logits)            # softmax
        grad = softmax.copy()                                # gradient of softmax
        grad[target_id] -= 1                                 # subtract 1 for target class
        h = self.W[input_id]                                 # get hidden representation
        self.U -= lr * np.outer(h, grad)                     # update output weights
        self.b -= lr * grad                                  # update bias
        self.W[input_id] -= lr * self.U @ grad               # update input embedding

# Create simple token-to-id mapping
all_tokens = set()                                           # set of all unique tokens
for ex in sft_data:                                          # iterate over examples
    for word in ex["instruction"].split():                   # tokenize instruction
        all_tokens.add(word.lower())                         # add lowercased word
    for word in ex["response"].split():                      # tokenize response
        all_tokens.add(word.lower())                         # add lowercased word

token_to_id = {t: i for i, t in enumerate(sorted(all_tokens))}  # map token -> id
id_to_token = {i: t for t, i in token_to_id.items()}        # map id -> token
vocab_size = len(token_to_id)                                # vocabulary size

print(f"SFT dataset: {len(sft_data)} примеров")              # print dataset size
print(f"Vocabulary size: {vocab_size} токенов")               # print vocab size

# Train the SFT model
model = SimpleSFTModel(vocab_size=vocab_size)                # initialize model
losses = []                                                  # list to store losses

for epoch in range(50):                                      # train for 50 epochs
    total_loss = 0                                           # accumulate loss
    for ex in sft_data:                                      # iterate over examples
        instr_tokens = ex["instruction"].split()             # tokenize instruction
        resp_tokens = ex["response"].split()                 # tokenize response
        # Simple: predict last word of response from first word of instruction
        inp_id = token_to_id.get(instr_tokens[0].lower(), 0)  # input token id
        tgt_id = token_to_id.get(resp_tokens[-1].lower(), 0)  # target token id
        loss = model.loss(inp_id, tgt_id)                    # compute loss
        model.train_step(inp_id, tgt_id, lr=0.05)           # train step
        total_loss += loss                                   # accumulate loss
    losses.append(total_loss / len(sft_data))                # average loss per epoch

print(f"Initial loss: {losses[0]:.4f}")                      # print initial loss
print(f"Final loss: {losses[-1]:.4f}")                       # print final loss
print("SFT обучение завершено!")

In [ ]:
# ============================================================
# STEP 3: Visualize SFT training curve
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))                     # create figure

ax.plot(range(1, 51), losses, 'b-', linewidth=2)            # plot loss curve
ax.set_xlabel('Эпоха', fontsize=12)                          # x-axis label
ax.set_ylabel('Loss (кросс-энтропия)', fontsize=12)          # y-axis label
ax.set_title('Обучение SFT: Loss по эпохам', fontsize=14)   # title
ax.axhline(y=losses[-1], color='red', linestyle='--',       # horizontal line at final loss
           alpha=0.5, label=f'Final loss: {losses[-1]:.4f}')  # label
ax.legend(fontsize=10)                                       # show legend

plt.tight_layout()                                           # adjust layout
plt.show()                                                   # display plot

print("SFT Training Curve: модель учится предсказывать правильные ответы")

---
## Шаг 4. Reward Model: обучение модели предпочтений

### Второй шаг RLHF пайплайна

Reward Model (RM) - это модель, которая **оценивает качество ответа** числом (reward score).

**Как создаётся:**
1. Для каждого промпта генерируем несколько ответов
2. Люди-аннотаторы ранжируют ответы (предпочтения)
3. Обучаем модель предсказывать предпочтения

### Модель Брэдли-Терри

Если ответ $y_w$ (winner) предпочтительнее $y_l$ (loser), то:

$$P(y_w > y_l | x) = \frac{\exp(r(x, y_w))}{\exp(r(x, y_w)) + \exp(r(x, y_l))} = \sigma(r(x, y_w) - r(x, y_l))$$

Где $r(x, y)$ - reward model, $\sigma$ - сигмоида.

### Loss функция

$$L_{RM} = -\mathbb{E}_{(x, y_w, y_l)} [\log \sigma(r(x, y_w) - r(x, y_l))]$$

Мы максимизируем вероятность того, что победитель получит больший reward.

---

In [ ]:
# ============================================================
# STEP 4: Reward Model training with Bradley-Terry model
# ============================================================

class RewardModel(nn.Module):                                # reward model class
    def __init__(self, input_dim=64, hidden_dim=32):        # constructor
        super().__init__()                                   # init parent class
        self.net = nn.Sequential(                            # sequential network
            nn.Linear(input_dim, hidden_dim),                # first linear layer
            nn.ReLU(),                                       # activation
            nn.Linear(hidden_dim, hidden_dim // 2),          # second linear layer
            nn.ReLU(),                                       # activation
            nn.Linear(hidden_dim // 2, 1)                    # output: scalar reward
        )

    def forward(self, x):                                    # forward pass
        return self.net(x).squeeze(-1)                       # return reward score

# Generate synthetic preference data
np.random.seed(42)                                           # for reproducibility
n_samples = 200                                              # number of preference pairs
input_dim = 64                                               # input dimension

# Create prompts as random vectors
prompts = torch.randn(n_samples, input_dim)                  # random prompt embeddings

# Create "good" and "bad" responses by adding noise
good_responses = prompts + torch.randn(n_samples, input_dim) * 0.3  # small noise = good
bad_responses = prompts + torch.randn(n_samples, input_dim) * 2.0   # large noise = bad

# Combine prompt and response as input to reward model
good_inputs = torch.cat([prompts, good_responses], dim=1)   # concat prompt + good response
bad_inputs = torch.cat([prompts, bad_responses], dim=1)     # concat prompt + bad response

# Initialize reward model
rm = RewardModel(input_dim=input_dim * 2, hidden_dim=64)    # model with doubled input dim
optimizer = torch.optim.Adam(rm.parameters(), lr=1e-3)      # Adam optimizer

# Train reward model with Bradley-Terry loss
rm_losses = []                                               # store losses
n_epochs = 100                                               # number of training epochs

for epoch in range(n_epochs):                                # training loop
    good_rewards = rm(good_inputs)                           # reward for good responses
    bad_rewards = rm(bad_inputs)                             # reward for bad responses
    # Bradley-Terry loss: -log(sigmoid(r_good - r_bad))
    diff = good_rewards - bad_rewards                        # reward difference
    loss = -torch.log(torch.sigmoid(diff) + 1e-8).mean()    # BT loss

    optimizer.zero_grad()                                    # zero gradients
    loss.backward()                                          # backpropagation
    optimizer.step()                                         # update weights

    rm_losses.append(loss.item())                            # store loss

    if (epoch + 1) % 25 == 0:                               # print every 25 epochs
        acc = (diff > 0).float().mean().item()               # accuracy: good > bad
        print(f"Epoch {epoch+1}: loss={loss.item():.4f}, accuracy={acc:.2%}")  # print metrics

print("\nReward Model обучен!")

In [ ]:
# ============================================================
# STEP 4: Visualize Reward Model training
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))             # create figure with 2 subplots

# Left: Training loss curve
ax = axes[0]                                                 # first subplot
ax.plot(range(1, n_epochs + 1), rm_losses, 'b-', linewidth=2)  # plot loss curve
ax.set_xlabel('Эпоха', fontsize=12)                          # x-axis label
ax.set_ylabel('Bradley-Terry Loss', fontsize=12)             # y-axis label
ax.set_title('Обучение Reward Model', fontsize=14)           # title
ax.axhline(y=rm_losses[-1], color='red', linestyle='--',    # final loss line
           alpha=0.5, label=f'Final: {rm_losses[-1]:.4f}')   # label
ax.legend(fontsize=10)                                       # legend

# Right: Reward distribution for good vs bad
ax2 = axes[1]                                                # second subplot
with torch.no_grad():                                        # no gradient computation
    good_scores = rm(good_inputs).numpy()                    # get good rewards
    bad_scores = rm(bad_inputs).numpy()                      # get bad rewards

ax2.hist(good_scores, bins=30, alpha=0.6, color='green',    # histogram of good rewards
         label='Хорошие ответы')                             # label
ax2.hist(bad_scores, bins=30, alpha=0.6, color='red',       # histogram of bad rewards
         label='Плохие ответы')                              # label
ax2.set_xlabel('Reward Score', fontsize=12)                  # x-axis label
ax2.set_ylabel('Количество', fontsize=12)                    # y-axis label
ax2.set_title('Распределение наград: хорошие vs плохие', fontsize=14)  # title
ax2.legend(fontsize=10)                                      # legend

plt.tight_layout()                                           # adjust layout
plt.show()                                                   # display plot

# Print separation quality
mean_good = np.mean(good_scores)                             # mean of good scores
mean_bad = np.mean(bad_scores)                               # mean of bad scores
print(f"Средний reward хороших ответов: {mean_good:.3f}")   # print mean good
print(f"Средний reward плохих ответов:  {mean_bad:.3f}")    # print mean bad
print(f"Разница: {mean_good - mean_bad:.3f}")               # print difference

---
## Шаг 5. Reward Model с нуля на numpy

Полностью запускаемая реализация модели наград. Каждая строка с комментарием.

Мы реализуем:
1. Простой двухслойный перцептрон как reward model
2. Модель Брэдли-Терри для обучения на предпочтениях
3. Sigmoid и её производную вручную
4. Обратное распространение ошибки вручную
5. SGD оптимизатор вручную

---

In [ ]:
# ============================================================
# STEP 5: Reward Model from scratch on numpy - FULLY RUNNABLE
# Every line commented, no black boxes
# ============================================================

# --- Activation functions ---

def sigmoid_numpy(x):                                        # sigmoid activation
    # Clip for numerical stability
    x = np.clip(x, -500, 500)                                # prevent overflow
    return 1.0 / (1.0 + np.exp(-x))                          # sigmoid formula

def sigmoid_derivative(x):                                   # derivative of sigmoid
    s = sigmoid_numpy(x)                                     # compute sigmoid
    return s * (1.0 - s)                                     # derivative: s * (1-s)

def relu_numpy(x):                                           # ReLU activation
    return np.maximum(0, x)                                  # max(0, x)

def relu_derivative(x):                                      # derivative of ReLU
    return (x > 0).astype(float)                             # 1 if x > 0, else 0

# --- Reward Model (2-layer MLP) ---

class NumpyRewardModel:                                      # reward model from scratch
    def __init__(self, input_dim, hidden_dim=16):            # constructor
        # Xavier initialization for weights
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)  # first layer weights
        self.b1 = np.zeros(hidden_dim)                       # first layer bias
        self.W2 = np.random.randn(hidden_dim, 1) * np.sqrt(2.0 / hidden_dim)  # second layer weights
        self.b2 = np.zeros(1)                                # second layer bias

    def forward(self, x):                                    # forward pass: x -> reward
        # x shape: (batch_size, input_dim)
        self.z1 = x @ self.W1 + self.b1                      # linear transform layer 1
        self.a1 = relu_numpy(self.z1)                        # ReLU activation
        self.z2 = self.a1 @ self.W2 + self.b2                # linear transform layer 2
        return self.z2.squeeze(-1)                            # reward score (batch_size,)

    def backward(self, x, grad_output):                      # backward pass
        batch_size = x.shape[0]                               # batch size
        # Gradient through output layer
        grad_z2 = grad_output.reshape(-1, 1)                 # reshape gradient: (batch, 1)
        grad_W2 = self.a1.T @ grad_z2 / batch_size           # gradient for W2
        grad_b2 = np.mean(grad_z2, axis=0)                   # gradient for b2
        # Gradient through hidden layer
        grad_a1 = grad_z2 @ self.W2.T                        # gradient flowing to a1
        grad_z1 = grad_a1 * relu_derivative(self.z1)         # gradient through ReLU
        grad_W1 = x.T @ grad_z1 / batch_size                 # gradient for W1
        grad_b1 = np.mean(grad_z1, axis=0)                   # gradient for b1
        return grad_W1, grad_b1, grad_W2, grad_b2            # return all gradients

    def update(self, grads, lr=0.001):                       # SGD update step
        grad_W1, grad_b1, grad_W2, grad_b2 = grads           # unpack gradients
        self.W1 -= lr * grad_W1                               # update W1
        self.b1 -= lr * grad_b1                               # update b1
        self.W2 -= lr * grad_W2                               # update W2
        self.b2 -= lr * grad_b2                               # update b2

# --- Bradley-Terry loss and its gradient ---

def bradley_terry_loss(r_w, r_l):                            # BT loss function
    # r_w: reward for winner, r_l: reward for loser
    diff = r_w - r_l                                         # difference in rewards
    return -np.mean(np.log(sigmoid_numpy(diff) + 1e-10))     # -log(sigma(r_w - r_l))

def bradley_terry_gradient(r_w, r_l):                        # gradient of BT loss
    diff = r_w - r_l                                         # difference in rewards
    # d/d(r_w) [-log(sigma(diff))] = -(1 - sigma(diff))
    grad_w = -(1.0 - sigmoid_numpy(diff))                    # gradient w.r.t. winner reward
    # d/d(r_l) [-log(sigma(diff))] = (1 - sigma(diff))
    grad_l = (1.0 - sigmoid_numpy(diff))                     # gradient w.r.t. loser reward
    return grad_w, grad_l                                    # return both gradients

# --- Generate synthetic preference data ---
np.random.seed(42)                                           # for reproducibility
n_pairs = 150                                                # number of preference pairs
input_dim = 32                                               # input dimension

# Each sample: concat of (prompt_embedding, response_embedding)
# Good responses are close to prompt, bad responses are far
X_winners = np.random.randn(n_pairs, input_dim)              # winner features
X_losers = X_winners + np.random.randn(n_pairs, input_dim) * 2.0  # loser: more noisy

# --- Train the Reward Model ---
rm_numpy = NumpyRewardModel(input_dim, hidden_dim=16)        # initialize model
lr = 0.01                                                    # learning rate
train_losses = []                                            # store losses
train_accuracies = []                                        # store accuracies

for epoch in range(100):                                     # training loop
    # Forward pass for winners and losers
    r_w = rm_numpy.forward(X_winners)                        # rewards for winners
    r_l = rm_numpy.forward(X_losers)                         # rewards for losers

    # Compute loss
    loss = bradley_terry_loss(r_w, r_l)                      # BT loss
    train_losses.append(loss)                                 # store loss

    # Compute accuracy
    acc = np.mean(r_w > r_l)                                 # fraction where winner > loser
    train_accuracies.append(acc)                              # store accuracy

    # Compute gradients from BT loss
    grad_w, grad_l = bradley_terry_gradient(r_w, r_l)       # BT gradients

    # Backprop through the network
    grads_w = rm_numpy.backward(X_winners, grad_w)           # backward for winners
    grads_l = rm_numpy.backward(X_losers, grad_l)            # backward for losers

    # Combine gradients (sum because both contribute to loss)
    combined_grads = (                                       # sum gradients from both
        grads_w[0] + grads_l[0],                             # W1 gradients
        grads_w[1] + grads_l[1],                             # b1 gradients
        grads_w[2] + grads_l[2],                             # W2 gradients
        grads_w[3] + grads_l[3]                              # b2 gradients
    )

    # Update weights
    rm_numpy.update(combined_grads, lr=lr)                   # SGD step

    if (epoch + 1) % 20 == 0:                               # print every 20 epochs
        print(f"Epoch {epoch+1}: loss={loss:.4f}, accuracy={acc:.2%}")  # print metrics

print("\nReward Model (numpy) обучен!")

In [ ]:
# ============================================================
# STEP 5: Visualize numpy Reward Model training
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))             # create figure with 3 subplots

# Plot 1: Loss curve
ax = axes[0]                                                 # first subplot
ax.plot(range(1, 101), train_losses, 'b-', linewidth=2)     # plot loss curve
ax.set_xlabel('Эпоха', fontsize=12)                          # x-axis label
ax.set_ylabel('Bradley-Terry Loss', fontsize=12)             # y-axis label
ax.set_title('Loss Reward Model (numpy)', fontsize=14)       # title
ax.axhline(y=train_losses[-1], color='red', linestyle='--', alpha=0.5)  # final loss line

# Plot 2: Accuracy curve
ax2 = axes[1]                                                # second subplot
ax2.plot(range(1, 101), train_accuracies, 'g-', linewidth=2)  # plot accuracy curve
ax2.set_xlabel('Эпоха', fontsize=12)                          # x-axis label
ax2.set_ylabel('Accuracy (winner > loser)', fontsize=12)      # y-axis label
ax2.set_title('Точность модели предпочтений', fontsize=14)    # title
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random baseline')  # baseline
ax2.axhline(y=train_accuracies[-1], color='green', linestyle='--', alpha=0.5, label=f'Final: {train_accuracies[-1]:.2%}')  # final
ax2.legend(fontsize=10)                                       # legend

# Plot 3: Reward distributions
ax3 = axes[2]                                                # third subplot
r_w_final = rm_numpy.forward(X_winners)                      # final winner rewards
r_l_final = rm_numpy.forward(X_losers)                       # final loser rewards
ax3.hist(r_w_final, bins=25, alpha=0.6, color='green',      # winner histogram
         label='Победители (хорошие)')                        # label
ax3.hist(r_l_final, bins=25, alpha=0.6, color='red',        # loser histogram
         label='Проигравшие (плохие)')                        # label
ax3.set_xlabel('Reward Score', fontsize=12)                   # x-axis label
ax3.set_ylabel('Количество', fontsize=12)                     # y-axis label
ax3.set_title('Распределение наград (numpy RM)', fontsize=14) # title
ax3.legend(fontsize=10)                                       # legend

plt.tight_layout()                                           # adjust layout
plt.show()                                                   # display plot

print(f"Финальная точность: {train_accuracies[-1]:.2%}")     # print final accuracy
print(f"Финальный loss: {train_losses[-1]:.4f}")             # print final loss

---
## Шаг 6. PPO: Proximal Policy Optimization

### Третий шаг RLHF пайплайна

PPO - алгоритм reinforcement learning, который **оптимизирует политику модели**, используя reward model как функцию награды.

### Зачем PPO, а не обычный градиентный спуск?

Проблема: если мы просто максимизируем reward, модель может:
1. **Слишком далеко уйти** от исходной SFT модели (mode collapse)
2. **Найти exploit** в reward model (reward hacking)
3. **Потерять разнообразие** (все ответы станут одинаковыми)

### Ключевые идеи PPO

**1. Clipped objective (обрезанная целевая функция):**

$$L^{CLIP} = \hat{E}_t [\min(r_t(\theta) \hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t)]$$

Где $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{old}(a_t|s_t)}$ - probability ratio

**2. KL штраф:** добавляем штраф за расхождение с исходной политикой

$$L_{total} = L^{CLIP} - \beta \cdot KL(\pi_\theta || \pi_{ref})$$

**3. Advantage function:** оцениваем, насколько действие лучше среднего

$$\hat{A}_t = R_t - V(s_t)$$

---

In [ ]:
# ============================================================
# STEP 6: PPO mathematics explained with code
# ============================================================

# PPO Core Components Explained Step by Step

# 1. Probability Ratio: how much the new policy differs from old
def compute_ratio(new_log_prob, old_log_prob):              # probability ratio
    # r(theta) = pi_new(a|s) / pi_old(a|s) = exp(log_pi_new - log_pi_old)
    return np.exp(new_log_prob - old_log_prob)               # exponentiate log-prob difference

# 2. Clipped Objective: prevent too large updates
def ppo_clipped_objective(ratio, advantage, epsilon=0.2):   # clipped surrogate objective
    # Clipped ratio: constrain ratio to [1-epsilon, 1+epsilon]
    clipped_ratio = np.clip(ratio, 1 - epsilon, 1 + epsilon)  # clip the ratio
    # Two objectives: unclipped and clipped
    obj_unclipped = ratio * advantage                        # unclipped objective
    obj_clipped = clipped_ratio * advantage                  # clipped objective
    # Take the minimum (pessimistic bound)
    return np.minimum(obj_unclipped, obj_clipped)            # min ensures conservative update

# 3. Advantage estimation
def compute_advantage(rewards, values, gamma=0.99):          # advantage = reward - baseline
    advantages = rewards - values                            # simple advantage: A = R - V(s)
    return advantages                                        # return advantages

# 4. KL divergence penalty
def kl_divergence_gaussian(mu_old, sigma_old, mu_new, sigma_new):  # KL between two Gaussians
    # KL(N(mu_old, sigma_old) || N(mu_new, sigma_new))
    kl = (np.log(sigma_new / sigma_old)                      # log ratio of stds
          + (sigma_old**2 + (mu_old - mu_new)**2) / (2 * sigma_new**2)  # quadratic term
          - 0.5)                                             # constant
    return kl                                                # return KL divergence

# Demonstrate with numerical examples
print("=" * 60)                                              # separator
print("PPO: КЛЮЧЕВЫЕ КОМПОНЕНТЫ")
print("=" * 60)                                              # separator

# Example 1: Probability ratio
old_lp = np.array([-1.5, -0.5, -2.0, -1.0])                # old log-probs
new_lp = np.array([-1.3, -0.7, -1.5, -2.0])                # new log-probs
ratios = compute_ratio(new_lp, old_lp)                       # compute ratios
print(f"\n1. Probability ratios: {ratios}")                 # print ratios
print(f"   ratio > 1: новая политика считает действие более вероятным")  # explanation
print(f"   ratio < 1: новая политика считает действие менее вероятным")  # explanation

# Example 2: Clipped objective
advantages = np.array([1.0, -0.5, 0.8, -1.0])              # advantages
epsilon = 0.2                                                # clip parameter
clipped_obj = ppo_clipped_objective(ratios, advantages, epsilon)  # compute clipped objective
print(f"\n2. Clipped objective (epsilon={epsilon}):")       # print header
print(f"   Ratios:    {ratios}")                             # print ratios
print(f"   Advantages: {advantages}")                        # print advantages
print(f"   Objective:  {clipped_obj}")                       # print objectives

# Example 3: Effect of epsilon
print(f"\n3. Эффект epsilon на clipping:")                  # print header
for eps in [0.1, 0.2, 0.3]:                                 # different epsilon values
    co = ppo_clipped_objective(np.array([1.5]), np.array([1.0]), eps)  # clipped objective
    print(f"   epsilon={eps}: ratio=1.5, advantage=1.0, objective={co[0]:.3f}")  # print result

# Example 4: KL penalty
print(f"\n4. KL дивергенция:")                              # print header
for shift in [0.0, 0.5, 1.0, 2.0]:                          # different mean shifts
    kl = kl_divergence_gaussian(0, 1, shift, 1)             # compute KL
    print(f"   shift={shift}: KL={kl:.4f}")                  # print KL value

print("\nPPO компоненты продемонстрированы!")

---
## Шаг 7. PPO пошагово на numpy

Полностью запускаемая реализация PPO с интерактивными виджетами.

Мы реализуем:
1. Простую политику (Gaussian policy)
2. Value function
3. PPO update с clipping
4. Визуализацию clipping функции

---

In [ ]:
# ============================================================
# STEP 7: PPO implementation from scratch on numpy
# With interactive widgets for epsilon visualization
# ============================================================

class GaussianPolicy:                                         # Gaussian policy for PPO
    def __init__(self, state_dim, action_dim=1):             # constructor
        self.state_dim = state_dim                           # state dimension
        self.action_dim = action_dim                         # action dimension
        # Policy parameters: mean and log_std
        self.W_mu = np.random.randn(state_dim, action_dim) * 0.1  # weights for mean
        self.b_mu = np.zeros(action_dim)                     # bias for mean
        self.log_std = np.zeros(action_dim)                  # log standard deviation

    def get_mu(self, states):                                # compute mean action
        return states @ self.W_mu + self.b_mu                # linear transform

    def get_std(self):                                       # get standard deviation
        return np.exp(self.log_std)                           # std = exp(log_std)

    def sample(self, states):                                # sample actions
        mu = self.get_mu(states)                             # compute mean
        std = self.get_std()                                 # compute std
        actions = mu + std * np.random.randn(*mu.shape)      # sample from Gaussian
        return actions, mu, std                              # return sample, mean, std

    def log_prob(self, actions, mu, std):                    # log probability of actions
        var = std ** 2                                       # variance
        log_p = -0.5 * np.log(2 * np.pi * var) - 0.5 * (actions - mu) ** 2 / var  # Gaussian log-prob
        return np.sum(log_p, axis=-1)                        # sum over action dims

class ValueFunction:                                         # value function V(s)
    def __init__(self, state_dim):                           # constructor
        self.W = np.random.randn(state_dim, 1) * 0.1        # weights
        self.b = np.zeros(1)                                 # bias

    def predict(self, states):                               # predict value
        return (states @ self.W + self.b).squeeze(-1)        # linear transform

    def update(self, states, targets, lr=0.01):             # update value function
        preds = self.predict(states)                          # current predictions
        error = preds - targets                               # compute error
        grad_W = states.T @ error.reshape(-1, 1) / len(states)  # gradient for W
        grad_b = np.mean(error)                               # gradient for b
        self.W -= lr * grad_W                                 # update W
        self.b -= lr * grad_b                                 # update b
        return np.mean(error ** 2)                            # return MSE loss

def ppo_update(policy, value_fn, states, actions, rewards,  # PPO update step
               old_log_probs, epsilon=0.2, beta_kl=0.1,     # PPO hyperparams
               lr_policy=0.01, lr_value=0.01, gamma=0.99):   # learning rates

    # Compute returns and advantages
    values = value_fn.predict(states)                         # V(s) for each state
    advantages = rewards - values                             # A(s) = R - V(s)
    advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)  # normalize

    # Current policy log-probs
    mu = policy.get_mu(states)                                # current mean
    std = policy.get_std()                                    # current std
    new_log_probs = policy.log_prob(actions, mu, std)         # current log-probs

    # Compute ratio: r(theta) = pi_new / pi_old
    ratio = np.exp(new_log_probs - old_log_probs)             # probability ratio

    # PPO clipped objective
    obj_unclipped = ratio * advantages                        # unclipped objective
    ratio_clipped = np.clip(ratio, 1 - epsilon, 1 + epsilon)  # clipped ratio
    obj_clipped = ratio_clipped * advantages                  # clipped objective
    ppo_objective = np.minimum(obj_unclipped, obj_clipped)    # take minimum

    # KL penalty
    mu_old = states @ policy.W_mu + policy.b_mu               # old mean (before update)
    kl = kl_divergence_gaussian(                              # compute KL divergence
        mu_old.mean(), policy.get_std().mean(),               # old distribution
        mu.mean(), std.mean()                                 # new distribution
    )

    # Total objective (maximize = minimize negative)
    total_objective = np.mean(ppo_objective) - beta_kl * kl   # add KL penalty

    # Policy gradient (simplified - compute gradient of ratio * advantage)
    grad_ratio = advantages * (1 - (ratio > 1 + epsilon).astype(float) - (ratio < 1 - epsilon).astype(float))  # gradient with clipping
    # Update policy weights
    policy.W_mu += lr_policy * states.T @ grad_ratio.reshape(-1, 1) / len(states)  # update mean weights
    # Update log_std (reduce if KL too high)
    if kl > 0.1:                                             # if KL is too high
        policy.log_std -= lr_policy * 0.1                     # increase std (more exploration)

    # Update value function
    value_loss = value_fn.update(states, rewards, lr=lr_value)  # value function update

    return {
        'ppo_objective': np.mean(ppo_objective),             # mean PPO objective
        'value_loss': value_loss,                             # value function loss
        'kl_divergence': kl,                                  # KL divergence
        'mean_ratio': np.mean(ratio),                         # mean probability ratio
        'mean_advantage': np.mean(advantages),                # mean advantage
        'clip_fraction': np.mean((ratio < 1 - epsilon) | (ratio > 1 + epsilon))  # fraction clipped
    }

print("PPO реализация готова!")

In [ ]:
# ============================================================
# STEP 7: Train PPO on a simple environment
# ============================================================

# Create a simple environment: maximize reward based on state
np.random.seed(42)                                           # for reproducibility
state_dim = 4                                                # state dimension

policy = GaussianPolicy(state_dim)                           # initialize policy
value_fn = ValueFunction(state_dim)                          # initialize value function

# Simulated environment: reward = -sum(state^2) + noise
# Goal: learn to produce actions that lead to low-magnitude states
def simulate_env(states, actions, noise_std=0.1):            # simple environment
    rewards = -np.sum(states ** 2, axis=1) + noise_std * np.random.randn(len(states))  # reward
    next_states = states + 0.1 * actions.reshape(-1, state_dim)  # state transition
    return next_states, rewards                               # return next state and reward

# Training loop
ppo_metrics = defaultdict(list)                               # store metrics
n_iterations = 50                                             # number of PPO iterations
batch_size = 32                                               # batch size

for iteration in range(n_iterations):                         # training loop
    # Collect trajectories
    states = np.random.randn(batch_size, state_dim) * 0.5    # random initial states
    actions, mu, std = policy.sample(states)                  # sample actions
    log_probs = policy.log_prob(actions, mu, std)             # log probabilities

    # Get rewards from environment
    next_states, rewards = simulate_env(states, actions)      # simulate environment

    # PPO update
    metrics = ppo_update(                                     # run PPO update
        policy, value_fn, states, actions, rewards,           # inputs
        log_probs,                                            # old log-probs
        epsilon=0.2, beta_kl=0.05,                           # PPO hyperparams
        lr_policy=0.01, lr_value=0.01                         # learning rates
    )

    # Store metrics
    for k, v in metrics.items():                              # iterate over metrics
        ppo_metrics[k].append(v)                              # store each metric

    if (iteration + 1) % 10 == 0:                            # print every 10 iterations
        print(f"Iter {iteration+1}: obj={metrics['ppo_objective']:.4f}, "
              f"KL={metrics['kl_divergence']:.4f}, "
              f"clip_frac={metrics['clip_fraction']:.2%}")

print("\nPPO обучение завершено!")

In [ ]:
# ============================================================
# STEP 7: Interactive PPO Clipping Visualization with Widgets
# ============================================================

from ipywidgets import interact, FloatSlider                  # import interactive widgets

def plot_ppo_clipping(epsilon=0.2):                          # function to plot PPO clipping
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))          # create figure

    # Plot 1: Clipped objective as function of ratio
    ax = axes[0]                                              # first subplot
    ratios = np.linspace(0.0, 3.0, 300)                      # range of probability ratios
    advantage_pos = 1.0                                       # positive advantage
    advantage_neg = -1.0                                      # negative advantage

    # For positive advantage
    obj_unclipped_pos = ratios * advantage_pos                # unclipped objective (pos)
    ratio_clipped_pos = np.clip(ratios, 1 - epsilon, 1 + epsilon)  # clipped ratio
    obj_clipped_pos = ratio_clipped_pos * advantage_pos       # clipped objective (pos)
    obj_ppo_pos = np.minimum(obj_unclipped_pos, obj_clipped_pos)  # PPO objective (pos)

    ax.plot(ratios, obj_unclipped_pos, 'b--', alpha=0.5,     # plot unclipped
            label='Unclipped (A>0)')                           # label
    ax.plot(ratios, obj_ppo_pos, 'b-', linewidth=2,          # plot PPO objective
            label='PPO clipped (A>0)')                         # label
    ax.axvline(x=1-epsilon, color='red', linestyle=':',      # left clip boundary
               alpha=0.7, label=f'1-eps = {1-epsilon:.2f}')   # label
    ax.axvline(x=1+epsilon, color='red', linestyle=':',      # right clip boundary
               alpha=0.7, label=f'1+eps = {1+epsilon:.2f}')   # label
    ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)  # zero line

    # For negative advantage
    obj_unclipped_neg = ratios * advantage_neg                # unclipped objective (neg)
    ratio_clipped_neg = np.clip(ratios, 1 - epsilon, 1 + epsilon)  # clipped ratio
    obj_clipped_neg = ratio_clipped_neg * advantage_neg       # clipped objective (neg)
    obj_ppo_neg = np.minimum(obj_unclipped_neg, obj_clipped_neg)  # PPO objective (neg)

    ax.plot(ratios, obj_unclipped_neg, 'g--', alpha=0.5,     # plot unclipped neg
            label='Unclipped (A<0)')                           # label
    ax.plot(ratios, obj_ppo_neg, 'g-', linewidth=2,          # plot PPO neg
            label='PPO clipped (A<0)')                         # label

    ax.set_xlabel('Probability Ratio r(theta)', fontsize=12)  # x-axis label
    ax.set_ylabel('Objective', fontsize=12)                    # y-axis label
    ax.set_title(f'PPO Clipping (epsilon={epsilon:.2f})', fontsize=14)  # title
    ax.legend(fontsize=9, loc='upper left')                    # legend
    ax.set_xlim(0, 3)                                         # x-axis range
    ax.set_ylim(-3, 3)                                        # y-axis range

    # Plot 2: Gradient of PPO objective
    ax2 = axes[1]                                              # second subplot
    # Gradient w.r.t. ratio: 1 if not clipped, 0 if clipped (for A>0)
    grad_pos = np.ones_like(ratios)                            # gradient starts as 1
    grad_pos[ratios > 1 + epsilon] = 0                        # zero where clipped (ratio too high)
    # For A<0: gradient is 1 when ratio in [1-eps, inf), 0 below
    grad_neg = np.ones_like(ratios)                            # gradient starts as 1
    grad_neg[ratios < 1 - epsilon] = 0                        # zero where clipped (ratio too low)

    ax2.plot(ratios, grad_pos, 'b-', linewidth=2,             # gradient for A>0
             label='Gradient (A>0)')                            # label
    ax2.plot(ratios, grad_neg, 'g-', linewidth=2,             # gradient for A<0
             label='Gradient (A<0)')                            # label
    ax2.axvline(x=1-epsilon, color='red', linestyle=':',      # left boundary
                alpha=0.7)                                      # style
    ax2.axvline(x=1+epsilon, color='red', linestyle=':',      # right boundary
                alpha=0.7)                                      # style
    ax2.fill_between(ratios, 0, 1, where=(ratios < 1-epsilon) | (ratios > 1+epsilon),  # clipped region
                     alpha=0.1, color='red', label='Clipped (no gradient)')  # label
    ax2.set_xlabel('Probability Ratio r(theta)', fontsize=12)  # x-axis label
    ax2.set_ylabel('Gradient', fontsize=12)                     # y-axis label
    ax2.set_title(f'PPO Gradient (epsilon={epsilon:.2f})', fontsize=14)  # title
    ax2.legend(fontsize=10)                                     # legend
    ax2.set_xlim(0, 3)                                          # x-axis range

    plt.tight_layout()                                          # adjust layout
    plt.show()                                                  # display plot

# Create interactive widget
interact(plot_ppo_clipping,                                    # function to call
         epsilon=FloatSlider(min=0.05, max=0.5, step=0.01,   # epsilon slider
                            value=0.2, description='Epsilon'));  # default value

In [ ]:
# ============================================================
# STEP 7: Visualize PPO training metrics
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))             # create 2x2 figure

# Plot 1: PPO Objective
ax = axes[0, 0]                                              # top-left subplot
ax.plot(range(1, n_iterations + 1), ppo_metrics['ppo_objective'], 'b-', linewidth=1.5)  # plot objective
ax.set_xlabel('Итерация', fontsize=11)                        # x-axis label
ax.set_ylabel('PPO Objective', fontsize=11)                   # y-axis label
ax.set_title('PPO Objective по итерациям', fontsize=13)      # title

# Plot 2: KL Divergence
ax = axes[0, 1]                                              # top-right subplot
ax.plot(range(1, n_iterations + 1), ppo_metrics['kl_divergence'], 'r-', linewidth=1.5)  # plot KL
ax.set_xlabel('Итерация', fontsize=11)                        # x-axis label
ax.set_ylabel('KL Divergence', fontsize=11)                   # y-axis label
ax.set_title('KL дивергенция (отклонение от ref)', fontsize=13)  # title
ax.axhline(y=0.1, color='orange', linestyle='--',            # target KL line
           alpha=0.5, label='Target KL')                      # label
ax.legend(fontsize=10)                                        # legend

# Plot 3: Clip Fraction
ax = axes[1, 0]                                              # bottom-left subplot
ax.plot(range(1, n_iterations + 1), ppo_metrics['clip_fraction'], 'g-', linewidth=1.5)  # plot clip fraction
ax.set_xlabel('Итерация', fontsize=11)                        # x-axis label
ax.set_ylabel('Clip Fraction', fontsize=11)                   # y-axis label
ax.set_title('Доля обрезанных ratio', fontsize=13)           # title
ax.set_ylim(0, 1)                                             # y-axis range

# Plot 4: Value Loss
ax = axes[1, 1]                                              # bottom-right subplot
ax.plot(range(1, n_iterations + 1), ppo_metrics['value_loss'], 'm-', linewidth=1.5)  # plot value loss
ax.set_xlabel('Итерация', fontsize=11)                        # x-axis label
ax.set_ylabel('Value Loss (MSE)', fontsize=11)                # y-axis label
ax.set_title('Value Function Loss', fontsize=13)              # title

plt.tight_layout()                                           # adjust layout
plt.show()                                                   # display plot
print("PPO метрики визуализированы!")

---
## Шаг 8. Полный пайплайн RLHF

### SFT -> RM -> PPO: три этапа выравнивания

1. **SFT (Supervised Fine-Tuning):** Обучаем модель на качественных примерах (промпт, ответ)
2. **RM (Reward Model):** Обучаем модель оценивать качество ответов по человеческим предпочтениям
3. **PPO (Reinforcement Learning):** Оптимизируем SFT модель, используя RM как награду

### Диаграмма пайплайна

---

In [ ]:
# ============================================================
# STEP 8: RLHF Pipeline Diagram
# ============================================================

fig, ax = plt.subplots(figsize=(16, 10))                     # create large figure
ax.set_xlim(0, 16)                                          # x-axis range
ax.set_ylim(0, 10)                                          # y-axis range
ax.axis('off')                                               # hide axes

# Define colors for each stage
colors = {                                                    # color scheme
    'pretrain': '#4A90D9',                                    # blue for pre-training
    'sft': '#7ED321',                                         # green for SFT
    'rm': '#F5A623',                                          # orange for RM
    'ppo': '#D0021B',                                         # red for PPO
    'output': '#9013FE',                                      # purple for output
    'arrow': '#333333',                                       # dark for arrows
    'human': '#50E3C2'                                        # teal for human data
}

# Stage 1: Pre-training
rect1 = plt.Rectangle((0.5, 7), 3, 2, facecolor=colors['pretrain'],  # pretraining box
                       edgecolor='black', linewidth=2, alpha=0.8)  # styling
ax.add_patch(rect1)                                           # add rectangle
ax.text(2, 8.3, 'PRE-TRAINING', fontsize=11, fontweight='bold',  # title
        ha='center', va='center', color='white')               # centered white text
ax.text(2, 7.5, 'Next-token\nprediction', fontsize=9,         # subtitle
        ha='center', va='center', color='white')               # centered white text

# Arrow from pre-training to SFT
ax.annotate('', xy=(4.5, 8), xytext=(3.5, 8),               # arrow coordinates
            arrowprops=dict(arrowstyle='->', color=colors['arrow'], lw=2))  # arrow style

# Stage 2: SFT
rect2 = plt.Rectangle((4.5, 7), 3, 2, facecolor=colors['sft'],  # SFT box
                       edgecolor='black', linewidth=2, alpha=0.8)  # styling
ax.add_patch(rect2)                                           # add rectangle
ax.text(6, 8.3, 'SFT', fontsize=11, fontweight='bold',        # title
        ha='center', va='center', color='white')               # centered white text
ax.text(6, 7.5, 'Instruction\nfollowing', fontsize=9,         # subtitle
        ha='center', va='center', color='white')               # centered white text

# Human data box for SFT
rect_h1 = plt.Rectangle((4.5, 5), 3, 1.2, facecolor=colors['human'],  # human data box
                         edgecolor='black', linewidth=1.5, alpha=0.7)  # styling
ax.add_patch(rect_h1)                                         # add rectangle
ax.text(6, 5.6, 'Человеческие\nпримеры', fontsize=9,          # label
        ha='center', va='center', color='black')               # centered text
ax.annotate('', xy=(6, 7), xytext=(6, 6.2),                 # arrow up to SFT
            arrowprops=dict(arrowstyle='->', color=colors['arrow'], lw=1.5))  # arrow style

# Arrow from SFT to RM
ax.annotate('', xy=(8.5, 8), xytext=(7.5, 8),               # arrow coordinates
            arrowprops=dict(arrowstyle='->', color=colors['arrow'], lw=2))  # arrow style

# Stage 3: Reward Model
rect3 = plt.Rectangle((8.5, 7), 3, 2, facecolor=colors['rm'],  # RM box
                       edgecolor='black', linewidth=2, alpha=0.8)  # styling
ax.add_patch(rect3)                                           # add rectangle
ax.text(10, 8.3, 'REWARD MODEL', fontsize=11, fontweight='bold',  # title
        ha='center', va='center', color='white')               # centered white text
ax.text(10, 7.5, 'Bradley-Terry\npreferences', fontsize=9,    # subtitle
        ha='center', va='center', color='white')               # centered white text

# Human data box for RM
rect_h2 = plt.Rectangle((8.5, 5), 3, 1.2, facecolor=colors['human'],  # human data box
                         edgecolor='black', linewidth=1.5, alpha=0.7)  # styling
ax.add_patch(rect_h2)                                         # add rectangle
ax.text(10, 5.6, 'Человеческие\nпредпочтения', fontsize=9,    # label
        ha='center', va='center', color='black')               # centered text
ax.annotate('', xy=(10, 7), xytext=(10, 6.2),               # arrow up to RM
            arrowprops=dict(arrowstyle='->', color=colors['arrow'], lw=1.5))  # arrow style

# Arrow from RM down and to PPO
ax.annotate('', xy=(6, 3.5), xytext=(10, 7),                # arrow from RM to PPO
            arrowprops=dict(arrowstyle='->', color=colors['rm'], lw=2,  # orange arrow
                           connectionstyle='arc3,rad=0.3'))    # curved arrow

# Stage 4: PPO (in the middle-bottom)
rect4 = plt.Rectangle((3, 2), 5, 2, facecolor=colors['ppo'],  # PPO box
                       edgecolor='black', linewidth=2, alpha=0.8)  # styling
ax.add_patch(rect4)                                           # add rectangle
ax.text(5.5, 3.3, 'PPO OPTIMIZATION', fontsize=11, fontweight='bold',  # title
        ha='center', va='center', color='white')               # centered white text
ax.text(5.5, 2.5, 'Clipped objective + KL penalty', fontsize=9,  # subtitle
        ha='center', va='center', color='white')               # centered white text

# Arrow from SFT down to PPO (reference policy)
ax.annotate('', xy=(5.5, 4), xytext=(6, 7),                 # arrow from SFT to PPO
            arrowprops=dict(arrowstyle='->', color=colors['sft'], lw=2,  # green arrow
                           connectionstyle='arc3,rad=-0.2', linestyle='dashed'))  # dashed
ax.text(4, 5.5, 'Ref policy\n(pi_ref)', fontsize=9,         # label for ref policy
        ha='center', va='center', color=colors['sft'],        # green text
        fontweight='bold')                                     # bold

# Arrow from RM to PPO (reward signal)
ax.text(8.5, 5.5, 'Reward\nsignal', fontsize=9,             # label for reward
        ha='center', va='center', color=colors['rm'],         # orange text
        fontweight='bold')                                     # bold

# Output: Aligned Model
rect5 = plt.Rectangle((11, 2), 3.5, 2, facecolor=colors['output'],  # output box
                       edgecolor='black', linewidth=2, alpha=0.8)  # styling
ax.add_patch(rect5)                                           # add rectangle
ax.text(12.75, 3.3, 'ALIGNED MODEL', fontsize=11, fontweight='bold',  # title
        ha='center', va='center', color='white')               # centered white text
ax.text(12.75, 2.5, 'Helpful + Honest\n+ Harmless', fontsize=9,  # subtitle
        ha='center', va='center', color='white')               # centered white text

# Arrow from PPO to Output
ax.annotate('', xy=(11, 3), xytext=(8, 3),                   # arrow from PPO to output
            arrowprops=dict(arrowstyle='->', color=colors['arrow'], lw=2.5))  # arrow style

# Title
ax.text(8, 9.7, 'RLHF Pipeline: SFT -> RM -> PPO',           # main title
        fontsize=16, fontweight='bold', ha='center', va='center')  # centered bold

# Add legend
legend_items = [                                               # legend entries
    (colors['pretrain'], 'Pre-training'),
    (colors['sft'], 'SFT'),
    (colors['rm'], 'Reward Model'),
    (colors['ppo'], 'PPO'),
    (colors['human'], 'Human Data'),
    (colors['output'], 'Aligned Model'),
]
for i, (color, label) in enumerate(legend_items):             # iterate over legend
    ax.add_patch(plt.Rectangle((0.5, 0.5 + i * 0.4), 0.3, 0.3,  # small color box
                               facecolor=color, alpha=0.8))    # fill color
    ax.text(1.0, 0.65 + i * 0.4, label, fontsize=9, va='center')  # label text

plt.tight_layout()                                           # adjust layout
plt.show()                                                   # display plot
print("Диаграмма RLHF пайплайна готова!")

---
## Шаг 9. DPO: Direct Preference Optimization

### Проблема RLHF

RLHF сложный:
- Нужна отдельная reward model
- Нужен PPO (нестабильный, много гиперпараметров)
- 4 модели одновременно: policy, ref, reward, value

### DPO: простая альтернатива

DPO (Rafailov et al., 2023) - **напрямую оптимизирует политику** на предпочтениях, **без reward model и PPO**.

### Ключевая идея

Из оптимальной политики RLHF можно вывести неявную reward:

$$r(x, y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)} + \beta \log Z(x)$$

Подставляя в Bradley-Terry, получаем DPO loss:

$$L_{DPO} = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)} \right) \right]$$

**Преимущества DPO:**
- Не нужна reward model
- Не нужен PPO
- Стабильнее обучения
- Меньше гиперпараметров

---

In [ ]:
# ============================================================
# STEP 9: DPO implementation from scratch on numpy
# Fully runnable, every line commented
# ============================================================

class DPOPolicy:                                              # DPO policy (Gaussian)
    def __init__(self, state_dim, action_dim=1):              # constructor
        self.state_dim = state_dim                            # state dimension
        self.action_dim = action_dim                          # action dimension
        self.W = np.random.randn(state_dim, action_dim) * 0.1  # policy weights
        self.b = np.zeros(action_dim)                         # policy bias
        self.log_std = np.zeros(action_dim)                   # log std

    def get_distribution(self, states):                       # get mean and std
        mu = states @ self.W + self.b                         # compute mean
        std = np.exp(self.log_std)                            # compute std
        return mu, std                                        # return params

    def log_prob(self, actions, mu, std):                     # log probability
        var = std ** 2                                        # variance
        log_p = -0.5 * np.log(2 * np.pi * var) - 0.5 * (actions - mu) ** 2 / var  # Gaussian
        return np.sum(log_p, axis=-1)                         # sum over action dims

# Create reference policy (frozen copy of initial policy)
def copy_policy(policy):                                      # deep copy of policy
    new_policy = DPOPolicy(policy.state_dim, policy.action_dim)  # create new policy
    new_policy.W = policy.W.copy()                            # copy weights
    new_policy.b = policy.b.copy()                            # copy bias
    new_policy.log_std = policy.log_std.copy()                # copy log_std
    return new_policy                                         # return copy

# DPO loss function
def dpo_loss(pi_log_w, ref_log_w, pi_log_l, ref_log_l, beta=0.1):  # DPO loss
    # pi_log_w: log pi_theta(y_w|x), ref_log_w: log pi_ref(y_w|x)
    # pi_log_l: log pi_theta(y_l|x), ref_log_l: log pi_ref(y_l|x)
    log_ratio_w = pi_log_w - ref_log_w                       # log(pi_theta / pi_ref) for winner
    log_ratio_l = pi_log_l - ref_log_l                       # log(pi_theta / pi_ref) for loser
    # DPO reward difference: beta * (log_ratio_w - log_ratio_l)
    reward_diff = beta * (log_ratio_w - log_ratio_l)         # scaled difference
    # Loss: -log(sigmoid(reward_diff))
    loss = -np.mean(np.log(sigmoid_numpy(reward_diff) + 1e-10))  # DPO loss
    return loss, reward_diff                                  # return loss and diff

# Generate synthetic preference data for DPO
np.random.seed(42)                                            # for reproducibility
n_dpo = 100                                                   # number of preference pairs
state_dim = 4                                                 # state dimension

states_dpo = np.random.randn(n_dpo, state_dim)                # random states
# Winner actions: close to zero (good behavior)
winner_actions = np.random.randn(n_dpo, 1) * 0.3              # good actions (small)
# Loser actions: far from zero (bad behavior)
loser_actions = np.random.randn(n_dpo, 1) * 2.0               # bad actions (large)

# Initialize policy and reference
pi_dpo = DPOPolicy(state_dim)                                 # trainable policy
pi_ref = copy_policy(pi_dpo)                                  # reference (frozen)

# DPO training
dpo_losses = []                                                # store losses
dpo_accuracies = []                                            # store accuracies
beta_dpo = 0.5                                                 # DPO beta parameter
lr_dpo = 0.01                                                  # learning rate

for epoch in range(100):                                       # training loop
    # Get log probabilities from current policy
    mu_w, std_w = pi_dpo.get_distribution(states_dpo)         # distribution for winners
    mu_l, std_l = pi_dpo.get_distribution(states_dpo)         # distribution for losers
    pi_log_w = pi_dpo.log_prob(winner_actions, mu_w, std_w)  # log pi_theta(y_w|x)
    pi_log_l = pi_dpo.log_prob(loser_actions, mu_l, std_l)   # log pi_theta(y_l|x)

    # Get log probabilities from reference policy (frozen)
    mu_ref_w, std_ref_w = pi_ref.get_distribution(states_dpo)  # ref distribution
    mu_ref_l, std_ref_l = pi_ref.get_distribution(states_dpo)  # ref distribution
    ref_log_w = pi_ref.log_prob(winner_actions, mu_ref_w, std_ref_w)  # log pi_ref(y_w|x)
    ref_log_l = pi_ref.log_prob(loser_actions, mu_ref_l, std_ref_l)   # log pi_ref(y_l|x)

    # Compute DPO loss
    loss, reward_diff = dpo_loss(pi_log_w, ref_log_w,        # DPO loss
                                  pi_log_l, ref_log_l, beta=beta_dpo)
    dpo_losses.append(loss)                                    # store loss
    acc = np.mean(reward_diff > 0)                             # accuracy
    dpo_accuracies.append(acc)                                 # store accuracy

    # Compute gradient (simplified for Gaussian policy)
    # Gradient of DPO loss w.r.t. policy mean
    beta = beta_dpo                                            # alias
    sig_diff = sigmoid_numpy(reward_diff)                      # sigmoid of reward diff
    # Gradient factor: (1 - sigmoid(diff)) * beta
    grad_factor_w = (1 - sig_diff) * beta / std_w             # gradient factor for winner
    grad_factor_l = (1 - sig_diff) * beta / std_l             # gradient factor for loser

    # Gradient of log_prob w.r.t. mu: (a - mu) / std^2
    # Combined gradient: -grad_factor_w * (winner - mu_w)/std_w^2 + grad_factor_l * (loser - mu_l)/std_l^2
    grad_w = -(1 - sig_diff) * beta * (winner_actions - mu_w) / (std_w ** 2)  # winner grad
    grad_l = (1 - sig_diff) * beta * (loser_actions - mu_l) / (std_l ** 2)    # loser grad

    # Update policy weights
    combined_grad = grad_w + grad_l                            # combine gradients
    pi_dpo.W -= lr_dpo * states_dpo.T @ combined_grad / n_dpo  # update W
    pi_dpo.b -= lr_dpo * np.mean(combined_grad, axis=0)       # update b

    if (epoch + 1) % 20 == 0:                                # print every 20 epochs
        print(f"Epoch {epoch+1}: loss={loss:.4f}, accuracy={acc:.2%}")  # print metrics

print("\nDPO обучение завершено!")

In [ ]:
# ============================================================
# STEP 9: Compare RLHF vs DPO visually
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))              # create figure with 3 subplots

# Plot 1: DPO Loss
ax = axes[0]                                                  # first subplot
ax.plot(range(1, 101), dpo_losses, 'purple', linewidth=2)    # plot DPO loss
ax.set_xlabel('Эпоха', fontsize=12)                           # x-axis label
ax.set_ylabel('DPO Loss', fontsize=12)                        # y-axis label
ax.set_title('DPO: Loss по эпохам', fontsize=14)              # title

# Plot 2: DPO Accuracy
ax2 = axes[1]                                                 # second subplot
ax2.plot(range(1, 101), dpo_accuracies, 'green', linewidth=2)  # plot accuracy
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5,  # random baseline
            label='Random baseline')                           # label
ax2.set_xlabel('Эпоха', fontsize=12)                          # x-axis label
ax2.set_ylabel('Accuracy (winner preferred)', fontsize=12)    # y-axis label
ax2.set_title('DPO: Точность предпочтений', fontsize=14)      # title
ax2.legend(fontsize=10)                                       # legend

# Plot 3: RLHF vs DPO comparison (conceptual)
ax3 = axes[2]                                                 # third subplot
methods = ['SFT', 'RLHF\n(PPO)', 'DPO']                      # method names
complexity = [1, 5, 2]                                        # implementation complexity
stability = [0.9, 0.5, 0.85]                                 # training stability
quality = [0.6, 0.9, 0.85]                                   # output quality

x_pos = np.arange(len(methods))                               # x positions
width = 0.25                                                  # bar width

bars1 = ax3.bar(x_pos - width, complexity, width,            # complexity bars
                label='Сложность', color='red', alpha=0.7)     # label
bars2 = ax3.bar(x_pos, stability, width,                     # stability bars
                label='Стабильность', color='blue', alpha=0.7)  # label
bars3 = ax3.bar(x_pos + width, quality, width,               # quality bars
                label='Качество', color='green', alpha=0.7)    # label

ax3.set_xticks(x_pos)                                         # set x ticks
ax3.set_xticklabels(methods, fontsize=11)                     # set x labels
ax3.set_ylabel('Оценка', fontsize=12)                         # y-axis label
ax3.set_title('SFT vs RLHF vs DPO', fontsize=14)             # title
ax3.legend(fontsize=10)                                       # legend

plt.tight_layout()                                            # adjust layout
plt.show()                                                    # display plot

# Print comparison table
print("\nСравнение методов выравнивания:")
print("-" * 60)
print(f"{'Метод':<15} {'Сложность':<15} {'Стабильность':<15} {'Качество':<15}")
print("-" * 60)
for i, method in enumerate(methods):                          # iterate over methods
    print(f"{method.replace(chr(10), ' '):<15} {complexity[i]:<15} {stability[i]:<15.2f} {quality[i]:<15.2f}")
print("-" * 60)

---
## Шаг 10. Интерактивный эксплорер выравнивания

Исследуйте, как параметры RLHF влияют на обучение и генерацию:

- **KL penalty (beta):** штраф за отклонение от reference policy
- **PPO clip range (epsilon):** насколько разрешено менять политику за один шаг
- **Reward scaling:** как масштаб награды влияет на обучение
- **DPO beta:** баланс между предпочтениями и reference policy

---

In [ ]:
# ============================================================
# STEP 10: Interactive Alignment Explorer with ipywidgets
# ============================================================

from ipywidgets import FloatSlider, VBox, HBox, Output, Dropdown  # import widget types
from IPython.display import clear_output                       # for clearing output

# Simulate the effect of hyperparameters on training
def simulate_rlhf_training(beta_kl=0.1, epsilon=0.2, reward_scale=1.0, n_steps=50):
    # Simulate training with given hyperparams
    np.random.seed(42)                                        # for reproducibility
    rewards_history = []                                       # track rewards
    kl_history = []                                            # track KL divergence
    diversity_history = []                                     # track response diversity

    current_reward = 0.0                                       # initial reward
    current_kl = 0.0                                           # initial KL
    current_diversity = 1.0                                    # initial diversity

    for step in range(n_steps):                                # training loop
        # Reward increases over time, but depends on reward_scale
        noise = np.random.randn() * 0.05                      # add noise
        current_reward += reward_scale * 0.02 + noise          # update reward

        # KL increases as policy moves away from reference
        # Higher beta_kl slows KL growth
        kl_increase = 0.02 / (1 + beta_kl * 10)              # KL increase rate
        current_kl += kl_increase + np.random.randn() * 0.005  # update KL

        # Diversity decreases as model converges
        # Higher reward_scale reduces diversity faster
        diversity_decay = 0.005 * reward_scale + 0.002        # decay rate
        current_diversity = max(0.1, current_diversity - diversity_decay)  # update diversity

        # Epsilon affects how quickly reward can grow
        # Larger epsilon = faster but potentially unstable learning
        if step > 0:                                           # after first step
            max_reward_step = epsilon * 0.5                    # max reward per step
            current_reward = min(current_reward,               # clip reward growth
                               current_reward)

        rewards_history.append(current_reward * reward_scale)  # store reward
        kl_history.append(current_kl)                          # store KL
        diversity_history.append(current_diversity)            # store diversity

    return rewards_history, kl_history, diversity_history      # return histories

def interactive_alignment_explorer(beta_kl=0.1, epsilon=0.2, reward_scale=1.0, dpo_beta=0.1):
    # Create the interactive plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))          # create 2x2 figure

    # Simulate RLHF training with current params
    rewards, kl, diversity = simulate_rlhf_training(          # run simulation
        beta_kl=beta_kl, epsilon=epsilon, reward_scale=reward_scale, n_steps=50)  # with params

    steps = range(1, 51)                                      # step numbers

    # Plot 1: Reward over training
    ax = axes[0, 0]                                            # top-left
    ax.plot(steps, rewards, 'g-', linewidth=2)                 # plot rewards
    ax.fill_between(steps, 0, rewards, alpha=0.1, color='green')  # fill area
    ax.set_xlabel('Шаг обучения', fontsize=11)                 # x label
    ax.set_ylabel('Reward', fontsize=11)                       # y label
    ax.set_title(f'Reward (scale={reward_scale:.1f})', fontsize=13)  # title

    # Plot 2: KL divergence over training
    ax2 = axes[0, 1]                                           # top-right
    ax2.plot(steps, kl, 'r-', linewidth=2)                     # plot KL
    ax2.axhline(y=0.1, color='orange', linestyle='--',        # target KL line
                alpha=0.7, label='Target KL')
    ax2.fill_between(steps, 0, kl, alpha=0.1, color='red')    # fill area
    ax2.set_xlabel('Шаг обучения', fontsize=11)                # x label
    ax2.set_ylabel('KL Divergence', fontsize=11)               # y label
    ax2.set_title(f'KL Div (beta_KL={beta_kl:.2f})', fontsize=13)  # title
    ax2.legend(fontsize=9)                                     # legend

    # Plot 3: Response diversity
    ax3 = axes[1, 0]                                           # bottom-left
    ax3.plot(steps, diversity, 'b-', linewidth=2)              # plot diversity
    ax3.fill_between(steps, 0, diversity, alpha=0.1, color='blue')  # fill area
    ax3.set_xlabel('Шаг обучения', fontsize=11)                # x label
    ax3.set_ylabel('Разнообразие ответов', fontsize=11)        # y label
    ax3.set_title('Diversity (влияние KL штрафа)', fontsize=13)  # title
    ax3.set_ylim(0, 1.1)                                       # y range

    # Plot 4: PPO clipping visualization with current epsilon
    ax4 = axes[1, 1]                                           # bottom-right
    ratios = np.linspace(0, 3, 300)                            # ratio range
    advantage = 1.0                                            # positive advantage
    obj_unclipped = ratios * advantage                         # unclipped objective
    obj_clipped = np.clip(ratios, 1 - epsilon, 1 + epsilon) * advantage  # clipped
    obj_ppo = np.minimum(obj_unclipped, obj_clipped)          # PPO objective

    ax4.plot(ratios, obj_unclipped, 'b--', alpha=0.5, label='Unclipped')  # unclipped
    ax4.plot(ratios, obj_ppo, 'b-', linewidth=2, label='PPO clipped')  # clipped
    ax4.axvline(x=1-epsilon, color='red', linestyle=':',      # left clip
                alpha=0.7, label=f'1-eps={1-epsilon:.2f}')
    ax4.axvline(x=1+epsilon, color='red', linestyle=':',      # right clip
                alpha=0.7, label=f'1+eps={1+epsilon:.2f}')
    ax4.set_xlabel('Probability Ratio', fontsize=11)           # x label
    ax4.set_ylabel('Objective', fontsize=11)                    # y label
    ax4.set_title(f'PPO Clipping (eps={epsilon:.2f})', fontsize=13)  # title
    ax4.legend(fontsize=8)                                      # legend
    ax4.set_xlim(0, 3)                                         # x range
    ax4.set_ylim(-0.5, 3.5)                                    # y range

    plt.tight_layout()                                         # adjust layout
    plt.show()                                                 # display plot

    # Print summary
    print(f"\nПараметры: beta_KL={beta_kl:.2f}, epsilon={epsilon:.2f}, "
          f"reward_scale={reward_scale:.1f}, dpo_beta={dpo_beta:.2f}")
    print(f"Финальный reward: {rewards[-1]:.3f}")
    print(f"Финальный KL: {kl[-1]:.4f}")
    print(f"Финальное разнообразие: {diversity[-1]:.3f}")

# Create interactive widget
interact(interactive_alignment_explorer,                       # function
         beta_kl=FloatSlider(min=0.01, max=1.0, step=0.01,   # KL penalty slider
                             value=0.1, description='beta_KL'),
         epsilon=FloatSlider(min=0.05, max=0.5, step=0.01,   # clip range slider
                            value=0.2, description='Epsilon'),
         reward_scale=FloatSlider(min=0.1, max=5.0, step=0.1,  # reward scale slider
                                  value=1.0, description='Reward scale'),
         dpo_beta=FloatSlider(min=0.01, max=1.0, step=0.01,  # DPO beta slider
                             value=0.1, description='DPO beta'));

---
## Шаг 11. FastAPI сервер: сравнение предпочтений

Запускаем полноценный сервер с reward model, где вы можете:
1. **Оценить ответ** - получить reward score
2. **Сравнить два ответа** - какой лучше и почему
3. **Запустить PPO шаг** - увидеть, как обновляется политика

### Инструкция

1. Запустите ячейку ниже - сервер запустится в фоне
2. Используйте следующие ячейки для отправки запросов
3. ngrok создаст публичный URL для доступа

---

In [ ]:
# ============================================================
# STEP 11: FastAPI server with Reward Model
# Student runs the server and interacts with it
# ============================================================

from fastapi import FastAPI                                   # FastAPI framework
from pydantic import BaseModel                                # data validation
import uvicorn                                                # ASGI server
import threading                                              # for background server

# Define request/response models
class ScoreRequest(BaseModel):                                # request for /score
    response: str                                             # response text to score

class ScoreResponse(BaseModel):                               # response for /score
    response: str                                             # original response
    reward_score: float                                       # reward score
    safety_score: float                                       # safety score
    helpfulness_score: float                                  # helpfulness score

class CompareRequest(BaseModel):                              # request for /compare
    response_a: str                                           # first response
    response_b: str                                           # second response

class CompareResponse(BaseModel):                             # response for /compare
    response_a: str                                           # first response
    response_b: str                                           # second response
    score_a: float                                            # reward for A
    score_b: float                                            # reward for B
    winner: str                                               # "A" or "B"
    confidence: float                                         # confidence level

class AlignRequest(BaseModel):                                # request for /align
    prompt: str                                               # input prompt
    response: str                                             # response to improve
    epsilon: float = 0.2                                      # PPO clip range
    beta_kl: float = 0.1                                     # KL penalty

class AlignResponse(BaseModel):                               # response for /align
    prompt: str                                               # original prompt
    original_response: str                                    # original response
    original_score: float                                     # original reward
    improved_score: float                                     # improved reward
    kl_divergence: float                                      # KL after update
    clip_fraction: float                                      # fraction of clipped ratios

# Simple text-based reward model (heuristics for demo)
class TextRewardModel:                                        # text-based reward model
    def __init__(self):                                       # constructor
        # Keywords associated with good responses
        self.helpful_keywords = [                             # helpful keywords
            'помог', 'решени', 'рекоменд', 'предлагаю',       # Russian helpful words
            'важно', 'обратите внимание', 'например',          # more helpful words
            'шаг', 'способ', 'метод', 'подход',                # method words
            'помощь', 'совет', 'объясняю', 'итак'             # more helpful words
        ]
        # Keywords associated with bad/dangerous responses
        self.harmful_keywords = [                             # harmful keywords
            'взлом', 'украсть', 'обман', 'мошенничеств',      # harmful words
            'незаконн', 'оружие', 'убить', 'нарко',            # more harmful words
            'уклонить', 'нарушить', 'обойти', 'причинить'      # more harmful words
        ]
        # Keywords for honest responses
        self.honest_keywords = [                              # honesty keywords
            'не знаю', 'не уверен', 'возможно', 'вероятно',    # uncertainty words
            'по моему мнению', 'насколько я знаю',             # hedging words
            'важно отметить', 'однако', 'с другой стороны'     # nuance words
        ]

    def score(self, text):                                    # compute reward score
        text_lower = text.lower()                             # lowercase text
        # Helpful score
        helpful = sum(1 for kw in self.helpful_keywords if kw in text_lower)  # count helpful
        helpful_score = min(1.0, helpful * 0.2)               # normalize to [0, 1]
        # Harmful score (inverse - more harmful = lower score)
        harmful = sum(1 for kw in self.harmful_keywords if kw in text_lower)  # count harmful
        safety_score = max(0.0, 1.0 - harmful * 0.3)          # more harmful = less safe
        # Honesty score
        honest = sum(1 for kw in self.honest_keywords if kw in text_lower)  # count honest
        honest_score = min(1.0, 0.3 + honest * 0.2)           # baseline + bonus
        # Length bonus (too short = bad, too long = okay)
        length_bonus = min(1.0, len(text) / 100) * 0.2       # length factor
        # Combined reward
        reward = helpful_score * 0.4 + safety_score * 0.35 + honest_score * 0.15 + length_bonus  # weighted
        return reward, safety_score, helpful_score             # return all scores

# Initialize reward model
reward_model = TextRewardModel()                               # create reward model instance

# Create FastAPI app
app = FastAPI(                                                 # create FastAPI app
    title="RLHF Reward Model Server",                          # app title
    description="Сервер для оценки и сравнения ответов",       # app description
    version="1.0"                                              # version
)

@app.get("/")                                                  # root endpoint
def root():                                                    # root handler
    return {                                                   # return info
        "message": "RLHF Reward Model Server",
        "endpoints": {
            "/score": "Оценить ответ (POST)",
            "/compare": "Сравнить два ответа (POST)",
            "/align": "Запустить PPO шаг (POST)",
            "/docs": "Документация API"
        }
    }

@app.post("/score", response_model=ScoreResponse)              # score endpoint
def score_response(request: ScoreRequest):                     # score handler
    reward, safety, helpfulness = reward_model.score(request.response)  # compute scores
    return ScoreResponse(                                      # return response
        response=request.response,                             # echo response
        reward_score=round(reward, 4),                         # reward score
        safety_score=round(safety, 4),                         # safety score
        helpfulness_score=round(helpfulness, 4)                # helpfulness score
    )

@app.post("/compare", response_model=CompareResponse)          # compare endpoint
def compare_responses(request: CompareRequest):                # compare handler
    score_a, _, _ = reward_model.score(request.response_a)     # score response A
    score_b, _, _ = reward_model.score(request.response_b)     # score response B
    diff = abs(score_a - score_b)                              # score difference
    confidence = min(1.0, diff * 5)                            # confidence from difference
    winner = "A" if score_a >= score_b else "B"                # determine winner
    return CompareResponse(                                    # return response
        response_a=request.response_a,                         # echo A
        response_b=request.response_b,                         # echo B
        score_a=round(score_a, 4),                             # score A
        score_b=round(score_b, 4),                             # score B
        winner=winner,                                         # winner
        confidence=round(confidence, 4)                        # confidence
    )

@app.post("/align", response_model=AlignResponse)              # align endpoint
def align_step(request: AlignRequest):                         # align handler
    original_score, _, _ = reward_model.score(request.response)  # original score
    # Simulate PPO step: slightly improve the score
    # In reality, this would update model weights
    improvement = request.epsilon * 0.1 * (1 - original_score)  # simulated improvement
    improved_score = min(1.0, original_score + improvement)    # improved score
    kl = np.random.uniform(0.01, 0.05) * (1 + request.beta_kl)  # simulated KL
    clip_frac = np.random.uniform(0.05, 0.2)                  # simulated clip fraction
    return AlignResponse(                                      # return response
        prompt=request.prompt,                                 # echo prompt
        original_response=request.response,                    # echo response
        original_score=round(original_score, 4),               # original score
        improved_score=round(improved_score, 4),               # improved score
        kl_divergence=round(kl, 4),                            # KL divergence
        clip_fraction=round(clip_frac, 4)                      # clip fraction
    )

print("FastAPI приложение создано!")                            # confirmation message
print("Эндпоинты: /score, /compare, /align")                  # list endpoints

In [ ]:
# ============================================================
# STEP 11: Start the FastAPI server with ngrok
# ============================================================

# Kill any existing server on port 8000
import os                                                      # for system commands
os.system('pkill -f uvicorn 2>/dev/null')                     # kill existing uvicorn

# Start server in background thread
def run_server():                                              # server runner function
    uvicorn.run(app, host="0.0.0.0", port=8000,              # run uvicorn
                log_level="warning")                           # minimal logging

server_thread = threading.Thread(target=run_server,           # create thread
                                daemon=True)                   # daemon = stops with main
server_thread.start()                                          # start server
time.sleep(2)                                                  # wait for server to start

# Start ngrok tunnel
from pyngrok import ngrok                                      # import ngrok

# Kill existing ngrok tunnels
ngrok.kill()                                                   # kill existing tunnels

# Create new tunnel
public_url = ngrok.connect(8000)                               # connect to port 8000
print(f"Сервер запущен!")
print(f"Публичный URL: {public_url}")
print(f"Документация: {public_url}/docs")
print("\nЭндпоинты:")
print(f"  POST {public_url}/score    - оценить ответ")
print(f"  POST {public_url}/compare  - сравнить два ответа")
print(f"  POST {public_url}/align    - запустить PPO шаг")

In [ ]:
# ============================================================
# STEP 11: Interact with the Reward Model server
# ============================================================

# Wait for server to be ready
time.sleep(2)                                                  # wait for server

# Get the ngrok URL
try:                                                           # try to get URL
    tunnels = ngrok.get_tunnels()                              # get active tunnels
    base_url = tunnels[0].public_url                           # get public URL
except:                                                        # fallback
    base_url = "http://localhost:8000"                         # local URL

print("Тестируем Reward Model сервер...")
print("=" * 60)

# Test 1: Score a good response
print("\n1. Оценка ХОРОШЕГО ответа:")
good_response = "Я могу помочь вам с этим вопросом. Вот пошаговое решение: сначала определите проблему, затем соберите данные, и наконец примените метод."  # good response
score_data = {"response": good_response}                       # request body
try:                                                           # try request
    resp = requests.post(f"{base_url}/score",                  # POST to /score
                        json=score_data, timeout=10)           # with JSON body
    result = resp.json()                                       # parse response
    print(f"   Ответ: {good_response[:60]}...")                # print truncated
    print(f"   Reward: {result['reward_score']}")              # print reward
    print(f"   Safety: {result['safety_score']}")              # print safety
    print(f"   Helpfulness: {result['helpfulness_score']}")    # print helpfulness
except Exception as e:                                         # handle errors
    print(f"   Ошибка: {e}")                                   # print error

# Test 2: Score a bad response
print("\n2. Оценка ПЛОХОГО ответа:")
bad_response = "Взломать аккаунт очень просто: нужно подобрать пароль и обойти защиту."  # bad response
score_data = {"response": bad_response}                        # request body
try:                                                           # try request
    resp = requests.post(f"{base_url}/score",                  # POST to /score
                        json=score_data, timeout=10)           # with JSON body
    result = resp.json()                                       # parse response
    print(f"   Ответ: {bad_response[:60]}...")                 # print truncated
    print(f"   Reward: {result['reward_score']}")              # print reward
    print(f"   Safety: {result['safety_score']}")              # print safety
    print(f"   Helpfulness: {result['helpfulness_score']}")    # print helpfulness
except Exception as e:                                         # handle errors
    print(f"   Ошибка: {e}")                                   # print error

# Test 3: Compare two responses
print("\n3. Сравнение двух ответов:")
compare_data = {                                               # compare request
    "response_a": "Столица Франции - Париж. Это один из самых красивых городов мира с богатой историей.",  # good response A
    "response_b": "Франция наверное где-то в Европе, а столица не помню, может Лион."  # bad response B
}
try:                                                           # try request
    resp = requests.post(f"{base_url}/compare",                # POST to /compare
                        json=compare_data, timeout=10)         # with JSON body
    result = resp.json()                                       # parse response
    print(f"   Ответ A (score={result['score_a']}): {compare_data['response_a'][:50]}...")  # print A
    print(f"   Ответ B (score={result['score_b']}): {compare_data['response_b'][:50]}...")  # print B
    print(f"   Победитель: {result['winner']}")                # print winner
    print(f"   Уверенность: {result['confidence']}")           # print confidence
except Exception as e:                                         # handle errors
    print(f"   Ошибка: {e}")                                   # print error

# Test 4: PPO alignment step
print("\n4. PPO шаг выравнивания:")
align_data = {                                                 # align request
    "prompt": "Расскажи о машинном обучении",                  # input prompt
    "response": "ML это когда компьютеры учатся сами, наверное.",  # initial response
    "epsilon": 0.2,                                            # clip range
    "beta_kl": 0.1                                             # KL penalty
}
try:                                                           # try request
    resp = requests.post(f"{base_url}/align",                  # POST to /align
                        json=align_data, timeout=10)           # with JSON body
    result = resp.json()                                       # parse response
    print(f"   Original score: {result['original_score']}")    # original score
    print(f"   Improved score: {result['improved_score']}")    # improved score
    print(f"   KL divergence: {result['kl_divergence']}")      # KL divergence
    print(f"   Clip fraction: {result['clip_fraction']}")      # clip fraction
except Exception as e:                                         # handle errors
    print(f"   Ошибка: {e}")                                   # print error

print("\nВсе тесты завершены!")

In [ ]:
# ============================================================
# STEP 11: Interactive server interaction with widgets
# ============================================================

def interact_with_server(response_text="Я могу помочь вам решить эту задачу. Вот мой подход к решению."):
    # Score the response
    try:                                                       # try request
        tunnels = ngrok.get_tunnels()                          # get tunnel
        base_url = tunnels[0].public_url                       # get URL
    except:                                                    # fallback
        base_url = "http://localhost:8000"                     # local URL

    score_data = {"response": response_text}                   # request body
    try:                                                       # try scoring
        resp = requests.post(f"{base_url}/score",              # POST request
                            json=score_data, timeout=10)       # with body
        result = resp.json()                                   # parse response

        # Create visual representation
        fig, ax = plt.subplots(figsize=(10, 4))               # create figure
        categories = ['Reward', 'Safety', 'Helpfulness']       # score categories
        scores = [result['reward_score'],                      # reward score
                  result['safety_score'],                       # safety score
                  result['helpfulness_score']]                  # helpfulness score
        colors = ['blue', 'green', 'orange']                   # bar colors

        bars = ax.barh(categories, scores, color=colors, alpha=0.7)  # horizontal bars
        ax.set_xlim(0, 1.1)                                   # x range
        ax.set_xlabel('Score', fontsize=12)                    # x label

        # Add score labels
        for bar, score in zip(bars, scores):                   # iterate bars
            ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,  # position
                    f'{score:.3f}', va='center', fontsize=11)   # score text

        ax.set_title(f'Оценка ответа: "{response_text[:40]}..."', fontsize=13)  # title
        plt.tight_layout()                                     # adjust layout
        plt.show()                                             # display plot

    except Exception as e:                                     # handle errors
        print(f"Ошибка: {e}")                                  # print error

# Create text input widget
text_input = widgets.Textarea(                                 # text area widget
    value='Я могу помочь вам решить эту задачу. Вот мой подход к решению.',  # default text
    placeholder='Введите ответ для оценки...',                 # placeholder
    description='Ответ:',                                      # label
    layout=widgets.Layout(width='600px', height='80px')        # size
)

# Create button
score_button = widgets.Button(                                 # button widget
    description='Оценить ответ',                               # button text
    button_style='success',                                    # green style
    layout=widgets.Layout(width='200px')                       # size
)

output = widgets.Output()                                      # output area

def on_score_click(b):                                         # button click handler
    with output:                                               # in output area
        output.clear_output()                                  # clear previous
        interact_with_server(text_input.value)                 # score the response

score_button.on_click(on_score_click)                          # connect handler

display(widgets.VBox([text_input, score_button, output]))     # display widgets
print("Введите текст ответа и нажмите 'Оценить ответ'")

---
## Шаг 12. Дашборд оценки безопасности

Визуализация метрик безопасности модели до и после выравнивания.

Метрики:
- **Safety Score:** доля безопасных ответов на вредоносные промпты
- **Refusal Rate:** доля отказов на опасные запросы
- **Helpfulness:** качество ответов на безопасные запросы
- **Bias Score:** уровень предвзятости в ответах

---

In [ ]:
# ============================================================
# STEP 12: Safety Evaluation Dashboard
# ============================================================

np.random.seed(42)                                              # for reproducibility

# Simulated safety metrics for different model versions
model_versions = ['Raw LM', 'SFT', 'RLHF (v1)', 'RLHF (v2)', 'DPO', 'Constitutional AI']  # model names

# Safety metrics (simulated but realistic)
metrics = {                                                     # metrics dictionary
    'Safety Score': [0.15, 0.55, 0.78, 0.87, 0.85, 0.93],    # safety scores
    'Refusal Rate': [0.02, 0.35, 0.65, 0.75, 0.72, 0.80],    # refusal rates
    'Helpfulness': [0.30, 0.70, 0.75, 0.82, 0.80, 0.85],     # helpfulness
    'Bias Score': [0.45, 0.35, 0.20, 0.12, 0.14, 0.08],      # bias (lower is better)
    'Hallucination': [0.50, 0.30, 0.18, 0.10, 0.12, 0.07],   # hallucination rate
}

# Create comprehensive dashboard
fig = plt.figure(figsize=(20, 16))                             # large figure

# Plot 1: All metrics radar chart (simplified as bar chart)
ax1 = fig.add_subplot(3, 2, 1)                                # position 1
x = np.arange(len(model_versions))                             # x positions
width = 0.15                                                   # bar width
for i, (metric, values) in enumerate(metrics.items()):        # iterate metrics
    offset = (i - 2) * width                                   # offset for each metric
    color = plt.cm.Set2(i / len(metrics))                      # color from colormap
    ax1.bar(x + offset, values, width, label=metric,          # plot bars
            color=color, alpha=0.8)                             # with alpha

ax1.set_xticks(x)                                              # x ticks
ax1.set_xticklabels(model_versions, rotation=30, ha='right', fontsize=9)  # x labels
ax1.set_ylabel('Score', fontsize=11)                           # y label
ax1.set_title('Все метрики по версиям модели', fontsize=13)   # title
ax1.legend(fontsize=8, loc='upper left')                       # legend
ax1.set_ylim(0, 1.1)                                           # y range

# Plot 2: Safety Score progression
ax2 = fig.add_subplot(3, 2, 2)                                # position 2
safety = metrics['Safety Score']                                # safety scores
colors_safety = plt.cm.RdYlGn(np.array(safety))               # color by value
bars = ax2.barh(model_versions, safety, color=colors_safety, alpha=0.8)  # horizontal bars
for i, (v, bar) in enumerate(zip(safety, bars)):              # add labels
    ax2.text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=10)  # score text
ax2.set_xlabel('Safety Score', fontsize=11)                    # x label
ax2.set_title('Safety Score по версиям', fontsize=13)         # title
ax2.set_xlim(0, 1.1)                                           # x range

# Plot 3: Safety vs Helpfulness trade-off
ax3 = fig.add_subplot(3, 2, 3)                                # position 3
scatter_colors = range(len(model_versions))                    # colors by version
scatter = ax3.scatter(metrics['Helpfulness'],                  # x = helpfulness
                       metrics['Safety Score'],                 # y = safety
                       c=scatter_colors, cmap='viridis',       # colormap
                       s=200, edgecolors='black', linewidth=1)  # size and edge
for i, name in enumerate(model_versions):                      # add labels
    ax3.annotate(name, (metrics['Helpfulness'][i],             # position
                        metrics['Safety Score'][i]),
                fontsize=8, ha='left', xytext=(5, 5),          # offset
                textcoords='offset points')                     # offset in points
ax3.set_xlabel('Helpfulness', fontsize=11)                     # x label
ax3.set_ylabel('Safety Score', fontsize=11)                    # y label
ax3.set_title('Trade-off: Safety vs Helpfulness', fontsize=13)  # title
ax3.set_xlim(0, 1)                                             # x range
ax3.set_ylim(0, 1)                                             # y range

# Plot 4: Refusal Rate on harmful prompts
ax4 = fig.add_subplot(3, 2, 4)                                # position 4
refusal = metrics['Refusal Rate']                               # refusal rates
ax4.fill_between(range(len(model_versions)), 0, refusal,       # area chart
                 alpha=0.3, color='red')                        # red fill
ax4.plot(range(len(model_versions)), refusal, 'r-o',           # line with markers
         linewidth=2, markersize=8)                             # styled
ax4.set_xticks(range(len(model_versions)))                     # x ticks
ax4.set_xticklabels(model_versions, rotation=30, ha='right', fontsize=9)  # x labels
ax4.set_ylabel('Refusal Rate', fontsize=11)                    # y label
ax4.set_title('Доля отказов на вредоносные промпты', fontsize=13)  # title
ax4.axhline(y=0.7, color='green', linestyle='--', alpha=0.5,  # target line
            label='Target (70%)')                               # label
ax4.legend(fontsize=9)                                         # legend

# Plot 5: Bias Score (lower is better)
ax5 = fig.add_subplot(3, 2, 5)                                # position 5
bias = metrics['Bias Score']                                    # bias scores
colors_bias = plt.cm.Reds(np.array(bias))                      # color by bias level
ax5.bar(model_versions, bias, color=colors_bias, alpha=0.8,    # bar chart
        edgecolor='black')                                      # with edge
for i, v in enumerate(bias):                                   # add labels
    ax5.text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=10)  # score text
ax5.set_ylabel('Bias Score (ниже = лучше)', fontsize=11)       # y label
ax5.set_title('Уровень предвзятости', fontsize=13)            # title
ax5.set_xticklabels(model_versions, rotation=30, ha='right', fontsize=9)  # x labels
ax5.axhline(y=0.1, color='green', linestyle='--', alpha=0.5,  # target line
            label='Target (<10%)')                              # label
ax5.legend(fontsize=9)                                         # legend

# Plot 6: Overall alignment quality
ax6 = fig.add_subplot(3, 2, 6)                                # position 6
# Compute overall alignment score (weighted average)
weights = {'Safety Score': 0.35, 'Refusal Rate': 0.15,        # weights for each metric
           'Helpfulness': 0.30, 'Bias Score': -0.1,           # bias is negative
           'Hallucination': -0.1}                              # hallucination is negative
overall = []                                                    # overall scores
for i in range(len(model_versions)):                           # iterate over models
    score = sum(metrics[m][i] * w for m, w in weights.items())  # weighted sum
    overall.append(max(0, score))                               # clip at 0

colors_overall = plt.cm.viridis(np.array(overall) / max(overall))  # color by score
bars = ax6.barh(model_versions, overall, color=colors_overall,  # horizontal bars
                alpha=0.8, edgecolor='black')                   # with edge
for i, v in enumerate(overall):                                # add labels
    ax6.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=10)  # score text
ax6.set_xlabel('Overall Alignment Score', fontsize=11)         # x label
ax6.set_title('Общая оценка выравнивания', fontsize=13)       # title

plt.tight_layout(pad=2.0)                                     # adjust layout with padding
plt.show()                                                     # display plot

print("Дашборд оценки безопасности готов!")

In [ ]:
# ============================================================
# STEP 12: Safety score distributions visualization
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))               # create figure

# Plot 1: Safety score distributions before and after RLHF
ax = axes[0]                                                   # first subplot
np.random.seed(42)                                             # for reproducibility
# Before RLHF: wide, low-scoring distribution
before_scores = np.random.beta(2, 5, 1000)                    # beta distribution (skewed low)
# After RLHF: narrow, high-scoring distribution
after_scores = np.random.beta(8, 2, 1000)                     # beta distribution (skewed high)

ax.hist(before_scores, bins=40, alpha=0.5, color='red',       # before histogram
        label='До RLHF', density=True)                         # label
ax.hist(after_scores, bins=40, alpha=0.5, color='green',      # after histogram
        label='После RLHF', density=True)                      # label
ax.axvline(x=np.mean(before_scores), color='red',             # mean before
           linestyle='--', linewidth=2,                        # dashed line
           label=f'Среднее до: {np.mean(before_scores):.2f}')  # label
ax.axvline(x=np.mean(after_scores), color='green',            # mean after
           linestyle='--', linewidth=2,                        # dashed line
           label=f'Среднее после: {np.mean(after_scores):.2f}')  # label
ax.set_xlabel('Safety Score', fontsize=12)                     # x label
ax.set_ylabel('Плотность', fontsize=12)                        # y label
ax.set_title('Распределение Safety Score: до и после RLHF', fontsize=13)  # title
ax.legend(fontsize=9)                                          # legend

# Plot 2: Category-wise safety improvement
ax2 = axes[1]                                                  # second subplot
categories = ['Физический\nвред', 'Мошенничество',           # harm categories
              'Предвзятость', 'Приватность',                   # more categories
              'Дезинформация', 'Самоповреждение']              # more categories
before_rates = [0.15, 0.20, 0.35, 0.25, 0.40, 0.10]          # safety rates before
after_rates = [0.90, 0.85, 0.82, 0.88, 0.78, 0.95]           # safety rates after

x = np.arange(len(categories))                                 # x positions
width = 0.35                                                   # bar width
ax2.bar(x - width/2, before_rates, width, label='До RLHF',    # before bars
        color='red', alpha=0.7)                                 # red
ax2.bar(x + width/2, after_rates, width, label='После RLHF',  # after bars
        color='green', alpha=0.7)                               # green

ax2.set_xticks(x)                                              # x ticks
ax2.set_xticklabels(categories, fontsize=9)                    # x labels
ax2.set_ylabel('Safety Rate', fontsize=12)                     # y label
ax2.set_title('Безопасность по категориям вреда', fontsize=13)  # title
ax2.legend(fontsize=10)                                        # legend
ax2.set_ylim(0, 1.1)                                           # y range

plt.tight_layout()                                             # adjust layout
plt.show()                                                     # display plot
print("Распределения безопасности визуализированы!")

---
## Шаг 13. Практические задания

Теперь ваша очередь! Вот 5 заданий для закрепления материала.

### Задание 1: Исследование гиперпараметров PPO

Измените параметры `epsilon`, `beta_kl` и `reward_scale` в интерактивном эксплорере (Шаг 10).
- Что происходит при `epsilon = 0.5`? При `epsilon = 0.05`?
- Как `beta_kl` влияет на KL дивергенцию и разнообразие?
- Найдите оптимальные параметры для максимального reward при KL < 0.1

### Задание 2: Реализация новой reward function

Модифицируйте `TextRewardModel` в FastAPI сервере (Шаг 11):
- Добавьте оценку **конкретности** ответа (есть ли числа, факты, примеры)
- Добавьте оценку **структурированности** (есть ли списки, заголовки)
- Добавьте штраф за **повторения** в тексте

### Задание 3: Сравнение RLHF и DPO

Используя реализации из блокнота:
- Обучите PPO и DPO на одних и тех же данных
- Сравните финальный reward, KL дивергенцию и стабильность
- В каких случаях DPO лучше RLHF? А когда наоборот?

### Задание 4: Red-teaming тестирование

Создайте набор из 10 вредоносных промптов и протестируйте reward model:
- Какие промпты получают высокий reward (ошибка модели)?
- Какие типы вреда труднее всего обнаружить?
- Предложите улучшения reward model для обнаружения этих случаев

### Задание 5: Реализация Constitutional AI

Реализуйте упрощённую версию Constitutional AI:
- Добавьте набор "принципов" (конституцию)
- Перед каждым ответом модель проверяет ответ по принципам
- Если ответ нарушает принцип, модель генерирует исправленную версию
- Сравните качество с RLHF и DPO

---

### Итоги блокнота

Вы изучили:
1. **Проблему выравнивания** - почему сырым LLM нужно выравнивание
2. **SFT** - supervised fine-tuning как первый шаг
3. **Reward Model** - модель Брэдли-Терри для предпочтений
4. **PPO** - proximal policy optimization для RLHF
5. **DPO** - прямая оптимизация предпочтений без RL
6. **Интерактивные виджеты** для исследования гиперпараметров
7. **FastAPI сервер** для hands-on работы с reward model
8. **Дашборд безопасности** для визуализации метрик

> Помните: выравнивание ИИ - одна из важнейших задач. Безопасный ИИ = полезный ИИ.

---